In [ ]:
from pathlib import Path
import os
import sys
_release_root = Path(os.environ.get('AVATAR_RELEASE_ROOT', Path.cwd())).resolve()
while not (_release_root / 'avatar_paths.py').is_file():
    if _release_root == _release_root.parent:
        raise RuntimeError('Open this notebook from the code package directory.')
    _release_root = _release_root.parent
if str(_release_root) not in sys.path:
    sys.path.insert(0, str(_release_root))
from avatar_paths import project_path, get_value, get_column, get_path, get_label


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from lifelines import KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test
plt.rcParams.update({'font.family': 'Arial', 'font.size': 14.5, 'axes.titlesize': 16.0, 'axes.labelsize': 15.0, 'xtick.labelsize': 14.0, 'ytick.labelsize': 14.0, 'legend.fontsize': 13.6, 'axes.linewidth': 1.2, 'xtick.major.width': 1.1, 'ytick.major.width': 1.1, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none', 'axes.grid': False, 'lines.linewidth': 2.1, 'lines.markersize': 8.2, 'errorbar.capsize': 4.2})
MBP_ORDER = ['MBP-I', 'MBP-II', 'MBP-III']
MBP_COLORS = {'MBP-I': '#154C83', 'MBP-II': '#9A6A18', 'MBP-III': '#8E2E2A'}
ENDPOINT_COLORS = {'Response': '#8E2E2A', 'PFS': '#1E5D8F', 'OS': '#50347A'}
COLOR_TEXT = '#111111'
COLOR_MUTED = '#333333'
COLOR_GRID = '#D0D0D0'
FIG_DPI = 600


In [ ]:
PROJECT_DIR = Path(str(project_path(get_value('science.paths.directory_008', 'directory_008'))))
MODEL_RESULTS_DIR = PROJECT_DIR / get_value('science.paths.directory_010', 'directory_010')
STAGE2_DIR = MODEL_RESULTS_DIR / get_value('science.paths.directory_011', 'directory_011')
COMBO_DIR = STAGE2_DIR / get_value('science.paths.directory_012', 'directory_012')
COMBO_CV_PRED_PATH = COMBO_DIR / get_value('science.tables.predictions_005', 'predictions_005.csv')
CLINICAL_PATH_CANDIDATES = [Path(str(project_path(get_value('science.tables.clinical_006', 'clinical_006.xlsx')))), Path(str(project_path(get_value('science.tables.clinical_001', 'clinical_001.xlsx'))))]

def first_existing(paths, label):
    for p in paths:
        if p.exists():
            
            print(p)
            return p
    raise FileNotFoundError(f'Cannot find {label}.\nChecked:\n' + '\n'.join((str(p) for p in paths)))
if not COMBO_CV_PRED_PATH.exists():
    raise FileNotFoundError(f"Cannot find the latest Stage2B repeated-CV prediction file:\n{COMBO_CV_PRED_PATH}{get_value('science.messages.status_023', '\n\nPlease confirm that the latest Stage2B combination analysis has been run under avatar_restart.')}")
CLINICAL_PATH = first_existing(CLINICAL_PATH_CANDIDATES, 'clinical baseline')
FIG5_OUT_DIR = STAGE2_DIR / get_value('science.paths.directory_073', 'directory_073')
MAIN_OUT_DIR = FIG5_OUT_DIR / 'Main'
SUPP_OUT_DIR = FIG5_OUT_DIR / 'Supplementary'
TABLE_OUT_DIR = FIG5_OUT_DIR / 'Tables'
SOURCE_OUT_DIR = FIG5_OUT_DIR / get_value('science.paths.directory_074', 'directory_074')
for p in [FIG5_OUT_DIR, MAIN_OUT_DIR, SUPP_OUT_DIR, TABLE_OUT_DIR, SOURCE_OUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)
print('Canonical project root:')

print('\nStage2 result root:')


print('1. Stage2B repeated-CV predictions:')

print('\n2. Clinical baseline:')





In [ ]:
def norm_id(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s.endswith('.0'):
        s = s[:-2]
    return s

def norm_name(x):
    return str(x).strip().lower().replace(' ', '').replace('_', '').replace('-', '')

def find_col(df, candidates, required=False):
    lookup = {norm_name(c): c for c in df.columns}
    for c in candidates:
        key = norm_name(c)
        if key in lookup:
            return lookup[key]
    if required:
        raise ValueError(f'Cannot find any of: {candidates}')
    return None

def numeric(df, candidates):
    c = find_col(df, candidates, required=False)
    if c is None:
        return pd.Series(np.nan, index=df.index, dtype=float)
    return pd.to_numeric(df[c], errors='coerce')

def binary(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).strip().lower()
    if s in {'1', 'yes', 'y', 'true', 'positive', 'present', 'r', 'responder', get_value('science.translation.analysis_001', 'present'), get_value('science.translation.analysis_002', 'yes'), get_value('science.translation.analysis_003', 'positive')}:
        return 1.0
    if s in {'0', 'no', 'n', 'false', 'negative', 'absent', 'nr', 'nonresponder', 'non-responder', get_value('science.translation.analysis_005', 'absent'), get_value('science.translation.analysis_006', 'no'), get_value('science.translation.analysis_007', 'negative')}:
        return 0.0
    try:
        return float(s)
    except Exception:
        return np.nan

def normalize_modalities(x):
    if pd.isna(x):
        return ''
    mapping = {'metagenomics': 'M', 'microbiome': 'M', 'ko': 'M', 'm': 'M', 'radiomics': 'R', 'ct radiomics': 'R', 'r': 'R', 'blood': 'B', 'blood metabolomics': 'B', 'b': 'B', 'urine': 'U', 'urine metabolomics': 'U', 'u': 'U'}
    raw = str(x).replace(',', '+').replace('|', '+').replace('/', '+')
    parts = [p.strip() for p in raw.split('+') if p.strip()]
    short = []
    for p in parts:
        key = p.lower().strip()
        if key in mapping:
            short.append(mapping[key])
        elif p in {'M', 'R', 'B', 'U'}:
            short.append(p)
    order = {'M': 0, 'R': 1, 'B': 2, 'U': 3}
    short = sorted(set(short), key=lambda z: order[z])
    return '+'.join(short)

def percentile_rank_01(series):
    s = pd.to_numeric(series, errors='coerce')
    if s.notna().sum() == 0:
        return pd.Series(np.nan, index=s.index)
    return s.rank(pct=True, method='average').clip(0, 1)
combo_pred = pd.read_csv(COMBO_CV_PRED_PATH, encoding='utf-8-sig')
print('Stage2B prediction shape:')
print(combo_pred.shape)
print('\nColumns:')
print(combo_pred.columns.tolist())
modality_col = find_col(combo_pred, ['modalities', 'modality', 'modality_combination', 'combination'], required=False)
model_col = find_col(combo_pred, ['model', 'model_name'], required=False)
if modality_col is not None:
    combo_pred['_modalities_short'] = combo_pred[modality_col].apply(normalize_modalities)
    mru_long = combo_pred[combo_pred['_modalities_short'] == 'M+R+U'].copy()
else:
    mru_long = pd.DataFrame()
if len(mru_long) == 0:
    if model_col is None:
        raise ValueError('Could not identify MRU: neither a usable modalities column nor a model column was found.')
    model_short = combo_pred[model_col].astype(str).apply(normalize_modalities)
    mru_long = combo_pred[model_short == 'M+R+U'].copy()
if len(mru_long) == 0:
    print('\nAvailable modality/model labels:')
    if modality_col is not None:
        print(combo_pred[modality_col].drop_duplicates().tolist())
    if model_col is not None:
        print(combo_pred[model_col].drop_duplicates().tolist())
    raise ValueError('MRU (M+R+U) could not be identified in the Stage2B prediction file.')
print('\nMRU repeated-CV rows:')
print(mru_long.shape)
if modality_col is not None:
    print('MRU modalities:', mru_long[modality_col].drop_duplicates().tolist())
if model_col is not None:
    print('MRU model label:', mru_long[model_col].drop_duplicates().tolist())
sample_col = find_col(mru_long, ['sample', 'sample_id', 'patient_id', 'ID', 'PatientID'], required=True)
fold_col = find_col(mru_long, ['fold_id', 'fold'], required=False)
rename_map = {sample_col: 'sample'}
required_prediction_candidates = {'Response_true': ['Response_true', 'Response', 'y_response'], 'Response_pred': ['Response_pred', 'Response_probability', 'response_prob'], 'PFS_time': ['PFS_time'], 'PFS_status': ['PFS_status', 'PFS_event'], 'PFS_risk': ['PFS_risk', 'PFS_pred'], 'OS_time': ['OS_time'], 'OS_status': ['OS_status', 'OS_event'], 'OS_risk': ['OS_risk', 'OS_pred']}
for canonical, candidates in required_prediction_candidates.items():
    c = find_col(mru_long, candidates, required=True)
    rename_map[c] = canonical
if fold_col is not None and fold_col != 'fold_id':
    rename_map[fold_col] = 'fold_id'
mru_long = mru_long.rename(columns=rename_map)
mru_long['sample'] = mru_long['sample'].map(norm_id)
numeric_prediction_cols = ['Response_true', 'Response_pred', 'PFS_time', 'PFS_status', 'PFS_risk', 'OS_time', 'OS_status', 'OS_risk']
for c in numeric_prediction_cols:
    mru_long[c] = pd.to_numeric(mru_long[c], errors='coerce')
mru_long = mru_long.dropna(subset=['sample', 'Response_true', 'Response_pred', 'PFS_time', 'PFS_status', 'PFS_risk', 'OS_time', 'OS_status', 'OS_risk']).copy()
consistency_rows = []
for outcome_col in ['Response_true', 'PFS_time', 'PFS_status', 'OS_time', 'OS_status']:
    counts = mru_long.groupby('sample')[outcome_col].nunique(dropna=True)
    n_inconsistent = int((counts > 1).sum())
    consistency_rows.append({'variable': outcome_col, 'n_patients': int(len(counts)), 'n_inconsistent': n_inconsistent})
cv_outcome_audit = pd.DataFrame(consistency_rows)
print('\nRepeated-CV outcome consistency audit:')

if (cv_outcome_audit['n_inconsistent'] > 0).any():
    raise ValueError('Outcome labels are not stable across repeated CV rows. Please inspect the Stage2B prediction file before rebuilding MBP.')
agg_spec = {'Response_true': 'first', 'Response_pred': 'mean', 'PFS_time': 'first', 'PFS_status': 'first', 'PFS_risk': 'mean', 'OS_time': 'first', 'OS_status': 'first', 'OS_risk': 'mean'}
mru_df = mru_long.groupby('sample', as_index=False).agg(agg_spec)
repeated_counts = mru_long.groupby('sample').size().rename('n_repeated_oof').reset_index()
mru_df = mru_df.merge(repeated_counts, on='sample', how='left')
print('\nPatient-level aggregated MRU predictions:')
print(mru_df.shape)
print('\nRepeated OOF count distribution:')
print(mru_df['n_repeated_oof'].value_counts().sort_index())
mru_df['Response_benefit'] = percentile_rank_01(mru_df['Response_pred'])
mru_df['PFS_benefit'] = 1 - percentile_rank_01(mru_df['PFS_risk'])
mru_df['OS_benefit'] = 1 - percentile_rank_01(mru_df['OS_risk'])
W_RESPONSE = 1 / 3
W_PFS = 1 / 3
W_OS = 1 / 3
mru_df['MRU_benefit_score'] = W_RESPONSE * mru_df['Response_benefit'] + W_PFS * mru_df['PFS_benefit'] + W_OS * mru_df['OS_benefit']
q_low = float(mru_df['MRU_benefit_score'].quantile(1 / 3))
q_high = float(mru_df['MRU_benefit_score'].quantile(2 / 3))

def assign_mbp(score):
    if score >= q_high:
        return 'MBP-I'
    if score >= q_low:
        return 'MBP-II'
    return 'MBP-III'
mru_df['MBP_raw'] = mru_df['MRU_benefit_score'].apply(assign_mbp)
mru_df['MBP'] = mru_df['MBP_raw'].map({'MBP-I': 'MBP-I\nHigh-benefit', 'MBP-II': 'MBP-II\nIntermediate', 'MBP-III': 'MBP-III\nLow-benefit'})
mru_df['MBP_risk_ord'] = mru_df['MBP_raw'].map({'MBP-I': 0, 'MBP-II': 1, 'MBP-III': 2})
mru_df = mru_df.sort_values('MRU_benefit_score', ascending=False).reset_index(drop=True)
print('\nRebuilt MBP tertile cutoffs:')
print(f'Lower tertile cutoff = {q_low:.6f}')
print(f'Upper tertile cutoff = {q_high:.6f}')
print('\nRebuilt MBP counts:')
rebuilt_counts = mru_df['MBP_raw'].value_counts().reindex(['MBP-I', 'MBP-II', 'MBP-III'])
print(rebuilt_counts)
if len(mru_df) != 109:
    warnings.warn(f'Expected 109 internal patients, but rebuilt MRU table contains {len(mru_df)}. Please confirm the locked Stage2B cohort.')
REBUILT_MBP_PATH = SOURCE_OUT_DIR / get_value('science.tables.result_087', 'result_087.csv')
mru_df.to_csv(REBUILT_MBP_PATH, index=False, encoding='utf-8-sig')
cv_outcome_audit.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_088', 'result_088.csv'), index=False, encoding='utf-8-sig')


clinical = pd.read_excel(CLINICAL_PATH)
clinical_sample_col = find_col(clinical, ['sample', 'sample_id', 'ID', 'PatientID', get_column('science.translation.analysis_009', 'patient_name')], required=True)
clinical = clinical.rename(columns={clinical_sample_col: 'sample'})
clinical['sample'] = clinical['sample'].map(norm_id)
clinical = clinical.dropna(subset=['sample']).drop_duplicates('sample').copy()
df = mru_df.merge(clinical, on='sample', how='left', suffixes=('', '_clinical'))
df['Response_observed'] = pd.to_numeric(df['Response_true'], errors='coerce')
df['AFP_plot'] = numeric(df, ['AFP'])
df['TBS_plot'] = numeric(df, ['TBS'])
if df['TBS_plot'].isna().all():
    size = numeric(df, ['Size', 'Tumor size', 'Tumor_size'])
    number = numeric(df, ['Number', 'Tumor number'])
    df['TBS_plot'] = np.sqrt(size ** 2 + number ** 2)
pvtt_col = find_col(df, ['PVTT'], required=False)
df['PVTT_plot'] = df[pvtt_col].apply(binary) if pvtt_col else np.nan
albi = numeric(df, ['ALBI', 'ALBI score', 'ALBI_score'])
if albi.isna().all():
    alb = numeric(df, ['ALB', 'Albumin'])
    tbil = numeric(df, ['TBiL', 'TBIL', 'Bilirubin'])
    if not alb.isna().all() and (not tbil.isna().all()):
        df['ALBI_score_plot'] = 0.66 * np.log10(tbil.clip(lower=0.1)) - 0.085 * alb
    else:
        df['ALBI_score_plot'] = numeric(df, ['ALBI_grade', 'ALBI grade'])
else:
    df['ALBI_score_plot'] = albi
n_with_clinical = int(df['AFP_plot'].notna().sum())
print('\nClinical table shape:')
print(clinical.shape)
print('\nMerged analysis table:')
print(df.shape)
print('\nPatients with non-missing AFP after merge:')
print(n_with_clinical)
if len(df) != len(mru_df):
    raise ValueError('Clinical merge changed the number of MRU patients.')
outcome_audit_rows = []
for base in ['Response_true', 'PFS_time', 'PFS_status', 'OS_time', 'OS_status']:
    clinical_candidates = {'Response_true': ['Response', 'mRECIST Response', 'mRECIST'], 'PFS_time': ['PFS_time'], 'PFS_status': ['PFS_status'], 'OS_time': ['OS_time'], 'OS_status': ['OS_status']}[base]
    if base == 'Response_true':
        clinical_col = find_col(clinical, clinical_candidates, required=False)
        if clinical_col is None:
            continue
        clin_small = clinical[['sample', clinical_col]].copy()
        clin_small['_clinical_value'] = clin_small[clinical_col].apply(binary)
    else:
        clinical_col = find_col(clinical, clinical_candidates, required=False)
        if clinical_col is None:
            continue
        clin_small = clinical[['sample', clinical_col]].copy()
        clin_small['_clinical_value'] = pd.to_numeric(clin_small[clinical_col], errors='coerce')
    audit_merge = mru_df[['sample', base]].merge(clin_small[['sample', '_clinical_value']], on='sample', how='inner')
    a = pd.to_numeric(audit_merge[base], errors='coerce')
    b = pd.to_numeric(audit_merge['_clinical_value'], errors='coerce')
    ok = a.notna() & b.notna()
    if base.endswith('_time'):
        mismatch = np.abs(a[ok] - b[ok]) > 1e-08
    else:
        mismatch = a[ok] != b[ok]
    outcome_audit_rows.append({'variable': base, 'n_compared': int(ok.sum()), 'n_mismatch': int(mismatch.sum())})
outcome_audit = pd.DataFrame(outcome_audit_rows)
print('\nMRU predictions vs clinical baseline outcome audit:')

outcome_audit.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_089', 'result_089.csv'), index=False, encoding='utf-8-sig')
print('\nFinal MBP counts used by Figure 5:')
display(df['MBP_raw'].value_counts().reindex(['MBP-I', 'MBP-II', 'MBP-III']).rename('n'))


In [ ]:
from scipy.stats import chi2_contingency
cross_5a = df[['sample', 'MBP_raw', 'Response_observed', 'MRU_benefit_score']].copy()
cross_5a['Response_observed'] = pd.to_numeric(cross_5a['Response_observed'], errors='coerce')
cross_5a = cross_5a.dropna(subset=['MBP_raw', 'Response_observed']).copy()
cross_5a['mRECIST'] = np.where(cross_5a['Response_observed'] == 1, 'R', 'NR')
MRECIST_ORDER = ['R', 'NR']
count_5a = pd.crosstab(cross_5a['mRECIST'], cross_5a['MBP_raw']).reindex(index=MRECIST_ORDER, columns=MBP_ORDER, fill_value=0)
row_pct_5a = count_5a.div(count_5a.sum(axis=1), axis=0) * 100
col_pct_5a = count_5a.div(count_5a.sum(axis=0), axis=1) * 100
chi2_5a, p_chi2_5a, dof_5a, expected_5a = chi2_contingency(count_5a.to_numpy())
expected_5a = pd.DataFrame(expected_5a, index=MRECIST_ORDER, columns=MBP_ORDER)
n_5a = int(count_5a.to_numpy().sum())
pearson_resid_5a = (count_5a - expected_5a) / np.sqrt(expected_5a)
row_prop_5a = count_5a.sum(axis=1) / n_5a
col_prop_5a = count_5a.sum(axis=0) / n_5a
asr_5a = pd.DataFrame(index=MRECIST_ORDER, columns=MBP_ORDER, dtype=float)
for r in MRECIST_ORDER:
    for c in MBP_ORDER:
        obs = float(count_5a.loc[r, c])
        exp = float(expected_5a.loc[r, c])
        denom = np.sqrt(exp * (1 - float(row_prop_5a.loc[r])) * (1 - float(col_prop_5a.loc[c])))
        if np.isfinite(denom) and denom > 0:
            asr_5a.loc[r, c] = (obs - exp) / denom
        else:
            asr_5a.loc[r, c] = np.nan
min_dim_5a = min(count_5a.shape[0] - 1, count_5a.shape[1] - 1)
if n_5a > 0 and min_dim_5a > 0:
    cramer_v_5a = float(np.sqrt(chi2_5a / (n_5a * min_dim_5a)))
else:
    cramer_v_5a = np.nan
cell_records_5a = []
for r in MRECIST_ORDER:
    for c in MBP_ORDER:
        cell_records_5a.append({'mRECIST': r, 'MBP': c, 'Observed_n': int(count_5a.loc[r, c]), 'Expected_n_under_independence': float(expected_5a.loc[r, c]), 'Row_percent': float(row_pct_5a.loc[r, c]), 'Column_percent': float(col_pct_5a.loc[r, c]), 'Pearson_residual': float(pearson_resid_5a.loc[r, c]), 'Adjusted_standardized_residual': float(asr_5a.loc[r, c])})
cell_summary_5a = pd.DataFrame(cell_records_5a)
association_summary_5a = pd.DataFrame([{'n': n_5a, 'chi_square': float(chi2_5a), 'degrees_of_freedom': int(dof_5a), 'chi_square_P': float(p_chi2_5a), 'Cramers_V': cramer_v_5a, 'minimum_expected_cell_count': float(expected_5a.to_numpy().min())}])
print('Observed MBP × mRECIST counts:')

print('\nExpected counts under independence:')

print('\nAdjusted standardized residuals:')

print('\nAssociation summary:')



In [ ]:
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
RESID_CMAP = LinearSegmentedColormap.from_list('association_residual', ['#C84E40', '#F0B7AE', '#F7F7F7', '#B7D2E6', '#2B6CB0'], N=256)
resid_mat_5a = asr_5a.loc[MRECIST_ORDER, MBP_ORDER].to_numpy(dtype=float)
max_abs_resid_5a = float(np.nanmax(np.abs(resid_mat_5a)))
RESID_LIM_5A = max(2.0, np.ceil(max_abs_resid_5a * 2) / 2)
resid_norm_5a = TwoSlopeNorm(vmin=-RESID_LIM_5A, vcenter=0, vmax=RESID_LIM_5A)
fig, ax = plt.subplots(figsize=(6.8, 3.85), facecolor='white')
im = ax.imshow(resid_mat_5a, cmap=RESID_CMAP, norm=resid_norm_5a, aspect='auto', interpolation='nearest')
ax.set_xticks(np.arange(-0.5, len(MBP_ORDER), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(MRECIST_ORDER), 1), minor=True)
ax.grid(which='minor', color='white', linewidth=3.1)
ax.tick_params(which='minor', bottom=False, left=False)
for i, r in enumerate(MRECIST_ORDER):
    for j, c in enumerate(MBP_ORDER):
        n_cell = int(count_5a.loc[r, c])
        pct_cell = float(row_pct_5a.loc[r, c])
        resid_cell = float(asr_5a.loc[r, c])
        text_color = 'white' if abs(resid_cell) >= 0.62 * RESID_LIM_5A else COLOR_TEXT
        ax.text(j, i - 0.065, f'{n_cell}  ({pct_cell:.1f}%)', ha='center', va='center', fontsize=15.2, fontweight='bold', color=text_color, zorder=3)
        ax.text(j, i + 0.185, f'ASR {resid_cell:+.2f}', ha='center', va='center', fontsize=11.7, color=text_color, alpha=0.92, zorder=3)
ax.set_xticks(np.arange(len(MBP_ORDER)))
ax.set_xticklabels(['MBP-I\nHigh-benefit', 'MBP-II\nIntermediate', 'MBP-III\nLow-benefit'])
for tick, g in zip(ax.get_xticklabels(), MBP_ORDER):
    tick.set_color(MBP_COLORS[g])
    tick.set_fontweight('bold')
row_n_5a = {r: int(count_5a.loc[r].sum()) for r in MRECIST_ORDER}
ax.set_yticks([0, 1])
ax.set_yticklabels([f"Responder (R)\nn={row_n_5a['R']}", f"Nonresponder (NR)\nn={row_n_5a['NR']}"])
ax.tick_params(axis='both', length=0, pad=8)
for spine in ax.spines.values():
    spine.set_visible(False)
cbar = fig.colorbar(im, ax=ax, orientation='vertical', fraction=0.052, pad=0.045)
cbar.set_label('Adjusted standardized residual', fontsize=12.9, labelpad=6)
cbar.ax.tick_params(labelsize=8.8, length=2.5)
cbar.outline.set_visible(False)
ax.text(1.0, 1.095, f"Cramér's V = {cramer_v_5a:.2f}", transform=ax.transAxes, ha='right', va='bottom', fontsize=12.6, color=COLOR_MUTED)
ax.text(0.0, 1.095, 'Blue: enriched   Red: depleted', transform=ax.transAxes, ha='left', va='bottom', fontsize=12, color=COLOR_MUTED)
fig.subplots_adjust(left=0.24, right=0.91, top=0.82, bottom=0.23)
STEM_5A = MAIN_OUT_DIR / get_value('science.paths.directory_075', 'directory_075')
fig.savefig(str(STEM_5A) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_5A) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_5A) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()
print(str(STEM_5A) + '.png')


In [ ]:
count_5a.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_090', 'result_090.csv'), encoding='utf-8-sig')
expected_5a.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_091', 'result_091.csv'), encoding='utf-8-sig')
row_pct_5a.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_092', 'result_092.csv'), encoding='utf-8-sig')
asr_5a.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_093', 'result_093.csv'), encoding='utf-8-sig')
cell_summary_5a.to_csv(SOURCE_OUT_DIR / get_value('science.tables.metrics_046', 'metrics_046.csv'), index=False, encoding='utf-8-sig')
association_summary_5a.to_csv(SOURCE_OUT_DIR / get_value('science.tables.metrics_047', 'metrics_047.csv'), index=False, encoding='utf-8-sig')
cross_5a.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_094', 'result_094.csv'), index=False, encoding='utf-8-sig')



In [ ]:
DAYS_PER_MONTH = 30.4375

def infer_survival_time_unit(values):
    x = pd.to_numeric(pd.Series(values), errors='coerce').dropna()
    if len(x) == 0:
        raise ValueError('Cannot infer survival-time unit from an empty series.')
    median_value = float(x.median())
    if median_value > 60:
        return ('days', median_value)
    return ('months', median_value)
pfs = df[['sample', 'MBP_raw', 'PFS_time', 'PFS_status']].copy()
pfs['PFS_time'] = pd.to_numeric(pfs['PFS_time'], errors='coerce')
pfs['PFS_status'] = pd.to_numeric(pfs['PFS_status'], errors='coerce')
pfs = pfs.dropna(subset=['MBP_raw', 'PFS_time', 'PFS_status']).copy()
pfs = pfs[pfs['PFS_time'] > 0].copy()
pfs['PFS_status'] = pfs['PFS_status'].astype(int)
PFS_INPUT_TIME_UNIT, PFS_RAW_MEDIAN = infer_survival_time_unit(pfs['PFS_time'])

print('Raw median PFS_time:', PFS_RAW_MEDIAN)
print('Raw PFS_time range:', (float(pfs['PFS_time'].min()), float(pfs['PFS_time'].max())))
if PFS_INPUT_TIME_UNIT == 'days':
    pfs['time_plot'] = pfs['PFS_time'] / DAYS_PER_MONTH
else:
    pfs['time_plot'] = pfs['PFS_time']
xlab = 'Progression-free survival (months)'
print('\nPFS after conversion to plotting months:')
print(pfs['time_plot'].describe(percentiles=[0.25, 0.5, 0.75]))
global_lr = multivariate_logrank_test(pfs['time_plot'], pfs['MBP_raw'], pfs['PFS_status'])
GLOBAL_P = float(global_lr.p_value)

def fmt_p(p):
    if not np.isfinite(p):
        return 'P=NA'
    if p < 0.001:
        return 'P<0.001'
    return f'P={p:.3f}'
summary_rows = []
for g in MBP_ORDER:
    d = pfs[pfs['MBP_raw'] == g].copy()
    km = KaplanMeierFitter().fit(d['time_plot'], event_observed=d['PFS_status'])
    med = float(km.median_survival_time_)
    summary_rows.append({'MBP': g, 'n': int(len(d)), 'PFS_events': int(d['PFS_status'].sum()), 'median_PFS_months': med if np.isfinite(med) else np.nan, 'global_logrank_P': GLOBAL_P, 'detected_input_time_unit': PFS_INPUT_TIME_UNIT})
summary_5b = pd.DataFrame(summary_rows)
print('\nFigure 5B summary:')



In [ ]:
max_time = float(pfs['time_plot'].max())
if max_time <= 36:
    tick_step = 6
elif max_time <= 72:
    tick_step = 12
else:
    tick_step = 24
xmax = np.ceil(max_time / tick_step) * tick_step
ticks = np.arange(0, xmax + 0.1, tick_step)
fig = plt.figure(figsize=(6.85, 5.85), facecolor='white')
gs = GridSpec(2, 1, figure=fig, height_ratios=[4.0, 1.18], hspace=0.07)
ax = fig.add_subplot(gs[0, 0])
axrisk = fig.add_subplot(gs[1, 0], sharex=ax)
legend_handles = []
legend_labels = []
km_parts = []
for g in MBP_ORDER:
    d = pfs[pfs['MBP_raw'] == g].copy()
    km = KaplanMeierFitter(label=g)
    km.fit(d['time_plot'], event_observed=d['PFS_status'])
    km.plot_survival_function(ax=ax, ci_show=True, ci_alpha=0.46, show_censors=True, color=MBP_COLORS[g], linewidth=2.1, censor_styles={'marker': '|', 'ms': 5.6, 'mew': 1.1})
    med = float(km.median_survival_time_)
    legend_handles.append(Line2D([0], [0], color=MBP_COLORS[g], lw=2.1))
    if np.isfinite(med):
        legend_labels.append(f'{g} (n={len(d)}, median {med:.1f} mo)')
    else:
        legend_labels.append(f'{g} (n={len(d)}, median NR)')
    sf = km.survival_function_.reset_index()
    sf.columns = ['time_months', 'survival_probability']
    ci = km.confidence_interval_.reset_index()
    ci.columns = ['time_months', 'CI_lower', 'CI_upper']
    sf = sf.merge(ci, on='time_months', how='left')
    sf['MBP'] = g
    km_parts.append(sf)
ax.legend(legend_handles, legend_labels, loc='lower center', bbox_to_anchor=(0.5, 1.03), frameon=False, fontsize=12.0, handlelength=2.8, ncol=1, borderaxespad=0.0)
ax.text(0.02, 0.96, 'Global log-rank ' + fmt_p(GLOBAL_P), transform=ax.transAxes, ha='left', va='top', fontsize=12.9, color=COLOR_TEXT, bbox=dict(boxstyle='round,pad=0.18', facecolor='white', edgecolor='none', alpha=0.92))
ax.set_xlim(0, xmax)
ax.set_ylim(0, 1.02)
ax.set_xticks(ticks)
ax.tick_params(axis='x', labelbottom=False)
ax.set_xlabel('')
ax.set_ylabel('Progression-free survival probability')
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
row_y = {'MBP-I': 2, 'MBP-II': 1, 'MBP-III': 0}
axrisk.set_xlim(0, xmax)
axrisk.set_ylim(-0.6, 2.6)
axrisk.set_xticks(ticks)
axrisk.set_xlabel(xlab)
axrisk.set_yticks([])
for s in axrisk.spines.values():
    s.set_visible(False)
axrisk.text(-0.02, 1.02, 'No. at risk', transform=axrisk.transAxes, ha='right', va='bottom', fontsize=12.6, fontweight='bold')
for g in MBP_ORDER:
    d = pfs[pfs['MBP_raw'] == g].copy()
    y = row_y[g]
    axrisk.text(-0.012, (y + 0.55) / 3.0, g, transform=axrisk.transAxes, ha='right', va='center', fontsize=12.3, fontweight='bold', color=MBP_COLORS[g])
    for t in ticks:
        n_risk = int((d['time_plot'] >= t).sum())
        axrisk.text(t, y, str(n_risk), ha='center', va='center', fontsize=12.3, color=COLOR_TEXT)
fig.subplots_adjust(left=0.14, right=0.98, top=0.82, bottom=0.1)
stem = MAIN_OUT_DIR / get_value('science.paths.directory_076', 'directory_076')
fig.savefig(str(stem) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(stem) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(stem) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()



In [ ]:
km_source_5b = pd.concat(km_parts, ignore_index=True)
summary_5b.to_csv(SOURCE_OUT_DIR / get_value('science.tables.metrics_048', 'metrics_048.csv'), index=False, encoding='utf-8-sig')
km_source_5b.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_095', 'result_095.csv'), index=False, encoding='utf-8-sig')
pfs.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_096', 'result_096.csv'), index=False, encoding='utf-8-sig')






In [ ]:
os_df = df[['sample', 'MBP_raw', 'OS_time', 'OS_status']].copy()
os_df['OS_time'] = pd.to_numeric(os_df['OS_time'], errors='coerce')
os_df['OS_status'] = pd.to_numeric(os_df['OS_status'], errors='coerce')
os_df = os_df.dropna(subset=['MBP_raw', 'OS_time', 'OS_status']).copy()
os_df = os_df[os_df['OS_time'] > 0].copy()
os_df['OS_status'] = os_df['OS_status'].astype(int)
OS_INPUT_TIME_UNIT, OS_RAW_MEDIAN = infer_survival_time_unit(os_df['OS_time'])

print('Raw median OS_time:', OS_RAW_MEDIAN)
print('Raw OS_time range:', (float(os_df['OS_time'].min()), float(os_df['OS_time'].max())))
if OS_INPUT_TIME_UNIT == 'days':
    os_df['time_plot'] = os_df['OS_time'] / DAYS_PER_MONTH
else:
    os_df['time_plot'] = os_df['OS_time']
OS_XLABEL = 'Overall survival (months)'
print('\nOS after conversion to plotting months:')
print(os_df['time_plot'].describe(percentiles=[0.25, 0.5, 0.75]))
global_lr_5c = multivariate_logrank_test(os_df['time_plot'], os_df['MBP_raw'], os_df['OS_status'])
GLOBAL_P_5C = float(global_lr_5c.p_value)
summary_rows_5c = []
for g in MBP_ORDER:
    d = os_df[os_df['MBP_raw'] == g].copy()
    km = KaplanMeierFitter().fit(d['time_plot'], event_observed=d['OS_status'])
    med = float(km.median_survival_time_)
    summary_rows_5c.append({'MBP': g, 'n': int(len(d)), 'OS_events': int(d['OS_status'].sum()), 'median_OS_months': med if np.isfinite(med) else np.nan, 'global_logrank_P': GLOBAL_P_5C, 'detected_input_time_unit': OS_INPUT_TIME_UNIT})
summary_5c = pd.DataFrame(summary_rows_5c)
print('\nFigure 5C summary:')



In [ ]:
max_time_5c = float(os_df['time_plot'].max())
if max_time_5c <= 36:
    tick_step_5c = 6
elif max_time_5c <= 72:
    tick_step_5c = 12
else:
    tick_step_5c = 24
xmax_5c = np.ceil(max_time_5c / tick_step_5c) * tick_step_5c
ticks_5c = np.arange(0, xmax_5c + 0.1, tick_step_5c)
fig = plt.figure(figsize=(6.85, 5.85), facecolor='white')
gs = GridSpec(2, 1, figure=fig, height_ratios=[4.0, 1.18], hspace=0.07)
ax = fig.add_subplot(gs[0, 0])
axrisk = fig.add_subplot(gs[1, 0], sharex=ax)
legend_handles_5c = []
legend_labels_5c = []
km_parts_5c = []
for g in MBP_ORDER:
    d = os_df[os_df['MBP_raw'] == g].copy()
    km = KaplanMeierFitter(label=g)
    km.fit(d['time_plot'], event_observed=d['OS_status'])
    km.plot_survival_function(ax=ax, ci_show=True, ci_alpha=0.46, show_censors=True, color=MBP_COLORS[g], linewidth=2.1, censor_styles={'marker': '|', 'ms': 5.6, 'mew': 1.1})
    med = float(km.median_survival_time_)
    legend_handles_5c.append(Line2D([0], [0], color=MBP_COLORS[g], linewidth=2.1))
    if np.isfinite(med):
        legend_labels_5c.append(f'{g} (n={len(d)}, median {med:.1f} mo)')
    else:
        legend_labels_5c.append(f'{g} (n={len(d)}, median NR)')
    sf = km.survival_function_.reset_index()
    sf.columns = ['time_months', 'survival_probability']
    ci = km.confidence_interval_.reset_index()
    ci.columns = ['time_months', 'CI_lower', 'CI_upper']
    sf = sf.merge(ci, on='time_months', how='left')
    sf['MBP'] = g
    km_parts_5c.append(sf)
ax.legend(legend_handles_5c, legend_labels_5c, loc='lower center', bbox_to_anchor=(0.5, 1.03), frameon=False, fontsize=12.0, handlelength=2.8, ncol=1, borderaxespad=0.0)
ax.text(0.02, 0.96, 'Global log-rank ' + fmt_p(GLOBAL_P_5C), transform=ax.transAxes, ha='left', va='top', fontsize=12.9, color=COLOR_TEXT, bbox=dict(boxstyle='round,pad=0.18', facecolor='white', edgecolor='none', alpha=0.92))
ax.set_xlim(0, xmax_5c)
ax.set_ylim(0, 1.02)
ax.set_xticks(ticks_5c)
ax.tick_params(axis='x', labelbottom=False)
ax.set_xlabel('')
ax.set_ylabel('Overall survival probability')
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
row_y_5c = {'MBP-I': 2, 'MBP-II': 1, 'MBP-III': 0}
axrisk.set_xlim(0, xmax_5c)
axrisk.set_ylim(-0.6, 2.6)
axrisk.set_xticks(ticks_5c)
axrisk.set_xlabel(OS_XLABEL)
axrisk.set_yticks([])
for s in axrisk.spines.values():
    s.set_visible(False)
axrisk.text(-0.02, 1.02, 'No. at risk', transform=axrisk.transAxes, ha='right', va='bottom', fontsize=12.6, fontweight='bold')
for g in MBP_ORDER:
    d = os_df[os_df['MBP_raw'] == g].copy()
    y = row_y_5c[g]
    axrisk.text(-0.012, (y + 0.55) / 3.0, g, transform=axrisk.transAxes, ha='right', va='center', fontsize=12.3, fontweight='bold', color=MBP_COLORS[g])
    for t in ticks_5c:
        n_risk = int((d['time_plot'] >= t).sum())
        axrisk.text(t, y, str(n_risk), ha='center', va='center', fontsize=12.3, color=COLOR_TEXT)
fig.subplots_adjust(left=0.14, right=0.98, top=0.82, bottom=0.1)
STEM_5C = MAIN_OUT_DIR / get_value('science.paths.directory_077', 'directory_077')
fig.savefig(str(STEM_5C) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_5C) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_5C) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()
print(str(STEM_5C) + '.png')


In [ ]:
km_source_5c = pd.concat(km_parts_5c, ignore_index=True)
summary_5c.to_csv(SOURCE_OUT_DIR / get_value('science.tables.metrics_049', 'metrics_049.csv'), index=False, encoding='utf-8-sig')
km_source_5c.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_097', 'result_097.csv'), index=False, encoding='utf-8-sig')
os_df.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_098', 'result_098.csv'), index=False, encoding='utf-8-sig')



In [ ]:
from scipy.stats import chi2
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from sklearn.model_selection import RepeatedStratifiedKFold
try:
    import statsmodels.api as sm
except ImportError as e:
    raise ImportError('Figure 5D requires statsmodels. Install with: pip install statsmodels') from e
clinical_model_df = df.copy()
clinical_model_df['MBP_worse'] = clinical_model_df['MBP_raw'].map({'MBP-I': 0, 'MBP-II': 1, 'MBP-III': 2}).astype(float)
clinical_model_df['mRECIST_R'] = pd.to_numeric(clinical_model_df['Response_observed'], errors='coerce')
clinical_model_df['Nonresponse'] = 1 - clinical_model_df['mRECIST_R']
clinical_model_df['ALBI_model'] = pd.to_numeric(clinical_model_df['ALBI_score_plot'], errors='coerce')
clinical_model_df['TBS_model'] = np.log1p(pd.to_numeric(clinical_model_df['TBS_plot'], errors='coerce').clip(lower=0))
clinical_model_df['AFP_raw_model'] = numeric(clinical_model_df, ['AFP'])
clinical_model_df['AFP_model'] = np.log1p(clinical_model_df['AFP_raw_model'].clip(lower=0))
clinical_model_df['NLR_raw_model'] = numeric(clinical_model_df, ['NLR'])
clinical_model_df['NLR_model'] = np.log1p(clinical_model_df['NLR_raw_model'].clip(lower=0))
clinical_model_df['PVTT_model'] = pd.to_numeric(clinical_model_df['PVTT_plot'], errors='coerce')

def zscore_safe(series):
    s = pd.to_numeric(series, errors='coerce')
    mean_val = float(s.mean())
    sd_val = float(s.std(ddof=0))
    if not np.isfinite(sd_val) or sd_val == 0:
        return pd.Series(0.0, index=s.index)
    return (s - mean_val) / sd_val
clinical_model_df['ALBI_z'] = zscore_safe(clinical_model_df['ALBI_model'])
clinical_model_df['TBS_z'] = zscore_safe(clinical_model_df['TBS_model'])
clinical_model_df['AFP_z'] = zscore_safe(clinical_model_df['AFP_model'])
clinical_model_df['NLR_z'] = zscore_safe(clinical_model_df['NLR_model'])
CLINICAL_ADJUSTERS_5D = ['ALBI_z', 'TBS_z', 'AFP_z', 'NLR_z', 'PVTT_model']
CLINICAL_BASE_COVARS_5F = ['ALBI_model', 'TBS_model', 'AFP_model', 'NLR_model', 'PVTT_model']
audit_cols_5def = ['MBP_worse', 'mRECIST_R', 'ALBI_model', 'TBS_model', 'AFP_model', 'NLR_model', 'PVTT_model', 'PFS_time', 'PFS_status', 'OS_time', 'OS_status']
clinical_model_audit_5def = pd.DataFrame({'variable': audit_cols_5def, 'n_nonmissing': [int(clinical_model_df[c].notna().sum()) for c in audit_cols_5def], 'n_missing': [int(clinical_model_df[c].isna().sum()) for c in audit_cols_5def]})

clinical_model_audit_5def.to_csv(SOURCE_OUT_DIR / get_value('science.tables.clinical_018', 'clinical_018.csv'), index=False, encoding='utf-8-sig')


In [ ]:
def fit_logistic_mbp_effect(data, adjusted=False):
    predictors = ['MBP_worse']
    if adjusted:
        predictors = predictors + CLINICAL_ADJUSTERS_5D
    d = data[['Nonresponse'] + predictors].dropna().copy()
    y = d['Nonresponse'].astype(int)
    X = sm.add_constant(d[predictors], has_constant='add')
    model = sm.GLM(y, X, family=sm.families.Binomial()).fit()
    beta = float(model.params['MBP_worse'])
    se = float(model.bse['MBP_worse'])
    p = float(model.pvalues['MBP_worse'])
    ci_low = beta - 1.959963984540054 * se
    ci_high = beta + 1.959963984540054 * se
    return {'Endpoint': 'Nonresponse', 'Effect_type': 'Adjusted' if adjusted else 'Unadjusted', 'Measure': 'OR', 'Estimate': float(np.exp(beta)), 'CI_lower': float(np.exp(ci_low)), 'CI_upper': float(np.exp(ci_high)), 'P': p, 'n': int(len(d)), 'events': int(y.sum())}

def fit_cox_mbp_effect(data, endpoint, adjusted=False):
    time_col = f'{endpoint}_time'
    status_col = f'{endpoint}_status'
    predictors = ['MBP_worse']
    if adjusted:
        predictors = predictors + CLINICAL_ADJUSTERS_5D
    d = data[[time_col, status_col] + predictors].dropna().copy()
    d[status_col] = d[status_col].astype(int)
    cph = CoxPHFitter()
    cph.fit(d, duration_col=time_col, event_col=status_col, show_progress=False)
    s = cph.summary.loc['MBP_worse']
    return {'Endpoint': endpoint, 'Effect_type': 'Adjusted' if adjusted else 'Unadjusted', 'Measure': 'HR', 'Estimate': float(s['exp(coef)']), 'CI_lower': float(s['exp(coef) lower 95%']), 'CI_upper': float(s['exp(coef) upper 95%']), 'P': float(s['p']), 'n': int(len(d)), 'events': int(d[status_col].sum())}
records_5d = []
for adjusted in [False, True]:
    records_5d.append(fit_logistic_mbp_effect(clinical_model_df, adjusted=adjusted))
    for endpoint in ['PFS', 'OS']:
        records_5d.append(fit_cox_mbp_effect(clinical_model_df, endpoint=endpoint, adjusted=adjusted))
result_5d = pd.DataFrame(records_5d)
endpoint_order_5d = {'Nonresponse': 0, 'PFS': 1, 'OS': 2}
effect_order_5d = {'Unadjusted': 0, 'Adjusted': 1}
result_5d['_endpoint_order'] = result_5d['Endpoint'].map(endpoint_order_5d)
result_5d['_effect_order'] = result_5d['Effect_type'].map(effect_order_5d)
result_5d = result_5d.sort_values(['_endpoint_order', '_effect_order']).drop(columns=['_endpoint_order', '_effect_order']).reset_index(drop=True)



In [ ]:
from matplotlib.ticker import ScalarFormatter
endpoint_labels_5d = {'Nonresponse': 'mRECIST non-responder', 'PFS': 'PFS', 'OS': 'OS'}
endpoint_colors_5d = {'Nonresponse': ENDPOINT_COLORS['Response'], 'PFS': ENDPOINT_COLORS['PFS'], 'OS': ENDPOINT_COLORS['OS']}
y_base_5d = {'Nonresponse': 2, 'PFS': 1, 'OS': 0}
offset_5d = {'Unadjusted': 0.17, 'Adjusted': -0.17}
fig, ax = plt.subplots(figsize=(7.25, 4.65), facecolor='white')
ax.axvline(1.0, color='#8C8C8C', linestyle='--', linewidth=1.1, zorder=0)
for y in [0, 1, 2]:
    ax.axhline(y, color=COLOR_GRID, linewidth=0.8, zorder=0)
for _, row in result_5d.iterrows():
    endpoint = row['Endpoint']
    effect_type = row['Effect_type']
    y = y_base_5d[endpoint] + offset_5d[effect_type]
    estimate = float(row['Estimate'])
    lo = float(row['CI_lower'])
    hi = float(row['CI_upper'])
    if effect_type == 'Unadjusted':
        point_face = 'white'
        point_edge = '#777777'
        line_color = '#A0A0A0'
        marker = 'o'
        zorder = 2
    else:
        point_face = endpoint_colors_5d[endpoint]
        point_edge = endpoint_colors_5d[endpoint]
        line_color = endpoint_colors_5d[endpoint]
        marker = 'D'
        zorder = 4
    ax.plot([lo, hi], [y, y], color=line_color, linewidth=1.6, zorder=zorder)
    ax.plot([lo, lo], [y - 0.045, y + 0.045], color=line_color, linewidth=1.2, zorder=zorder)
    ax.plot([hi, hi], [y - 0.045, y + 0.045], color=line_color, linewidth=1.2, zorder=zorder)
    ax.scatter(estimate, y, s=36 if effect_type == 'Adjusted' else 28, marker=marker, facecolor=point_face, edgecolor=point_edge, linewidth=1.2, zorder=5)
    if effect_type == 'Adjusted':
        p = float(row['P'])
        if p < 0.001:
            p_txt = 'P<0.001'
        else:
            p_txt = f'P={p:.3f}'
        ax.text(hi * 1.1, y, f'{estimate:.2f} ({lo:.2f}–{hi:.2f}); {p_txt}', ha='left', va='center', fontsize=12.1, color=COLOR_TEXT)
positive_lows_5d = result_5d['CI_lower'].replace([np.inf, -np.inf], np.nan).dropna()
positive_highs_5d = result_5d['CI_upper'].replace([np.inf, -np.inf], np.nan).dropna()
xmin_5d = max(0.15, float(positive_lows_5d.min()) / 1.8)
xmax_5d = min(100.0, max(5.0, float(positive_highs_5d.max()) * 2.3))
ax.set_xscale('log')
ax.set_xlim(xmin_5d, xmax_5d)
ax.set_yticks([2, 1, 0])
ax.set_yticklabels([endpoint_labels_5d['Nonresponse'], endpoint_labels_5d['PFS'], endpoint_labels_5d['OS']])
ax.set_xlabel('Effect per one-level worsening of MBP (OR / HR, log scale)')
ax.tick_params(axis='y', length=0, pad=8)
ax.grid(False)
for spine in ['top', 'right', 'left']:
    ax.spines[spine].set_visible(False)
legend_handles_5d = [Line2D([0], [0], marker='o', color='#A0A0A0', markerfacecolor='white', markeredgecolor='#777777', linewidth=1.4, label='Unadjusted'), Line2D([0], [0], marker='D', color=COLOR_TEXT, markerfacecolor=COLOR_TEXT, linewidth=1.4, label='Adjusted')]
ax.legend(handles=legend_handles_5d, loc='upper left', frameon=False, fontsize=12.5, ncol=2, columnspacing=1.4, handlelength=2)
fig.subplots_adjust(left=0.24, right=0.8, top=0.84, bottom=0.12)
STEM_5D = MAIN_OUT_DIR / get_value('science.paths.directory_078', 'directory_078')
fig.savefig(str(STEM_5D) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_5D) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_5D) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
result_5d.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_099', 'result_099.csv'), index=False, encoding='utf-8-sig')



In [ ]:
def fit_within_mrecist_cox(data, mrecist_value, endpoint):
    time_col = f'{endpoint}_time'
    status_col = f'{endpoint}_status'
    d = data[data['mRECIST_R'] == mrecist_value][[time_col, status_col, 'MBP_worse']].dropna().copy()
    d[status_col] = d[status_col].astype(int)
    cph = CoxPHFitter()
    cph.fit(d, duration_col=time_col, event_col=status_col, show_progress=False)
    s = cph.summary.loc['MBP_worse']
    return {'mRECIST': 'Responder' if mrecist_value == 1 else 'Nonresponder', 'mRECIST_code': int(mrecist_value), 'Endpoint': endpoint, 'HR_per_MBP_level_worsening': float(s['exp(coef)']), 'CI_lower': float(s['exp(coef) lower 95%']), 'CI_upper': float(s['exp(coef) upper 95%']), 'P': float(s['p']), 'n': int(len(d)), 'events': int(d[status_col].sum())}
records_5e = []
for mrecist_value in [1, 0]:
    for endpoint in ['PFS', 'OS']:
        records_5e.append(fit_within_mrecist_cox(clinical_model_df, mrecist_value=mrecist_value, endpoint=endpoint))
result_5e = pd.DataFrame(records_5e)



In [ ]:
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
MRECIST_DISPLAY_5E = ['Responder', 'Nonresponder']
ENDPOINT_ORDER_5E = ['PFS', 'OS']
log2hr_5e = np.full((len(ENDPOINT_ORDER_5E), len(MRECIST_DISPLAY_5E)), np.nan, dtype=float)
lookup_5e = {}
for i, endpoint in enumerate(ENDPOINT_ORDER_5E):
    for j, mrecist in enumerate(MRECIST_DISPLAY_5E):
        row = result_5e[(result_5e['Endpoint'] == endpoint) & (result_5e['mRECIST'] == mrecist)]
        if len(row) != 1:
            continue
        r = row.iloc[0]
        hr = float(r['HR_per_MBP_level_worsening'])
        log2hr_5e[i, j] = np.log2(hr)
        lookup_5e[endpoint, mrecist] = r
EFFECT_CMAP_5E = LinearSegmentedColormap.from_list('within_mrecist_effect', ['#2B6CB0', '#B7D2E6', '#F7F7F7', '#F1B2A8', '#C84E40'], N=256)
max_abs_5e = float(np.nanmax(np.abs(log2hr_5e)))
EFFECT_LIM_5E = max(1.0, np.ceil(max_abs_5e * 2) / 2)
norm_5e = TwoSlopeNorm(vmin=-EFFECT_LIM_5E, vcenter=0, vmax=EFFECT_LIM_5E)
fig, ax = plt.subplots(figsize=(6.5, 3.75), facecolor='white')
im = ax.imshow(log2hr_5e, cmap=EFFECT_CMAP_5E, norm=norm_5e, aspect='auto', interpolation='nearest')
ax.set_xticks(np.arange(-0.5, len(MRECIST_DISPLAY_5E), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(ENDPOINT_ORDER_5E), 1), minor=True)
ax.grid(which='minor', color='white', linewidth=3.3)
ax.tick_params(which='minor', bottom=False, left=False)
for i, endpoint in enumerate(ENDPOINT_ORDER_5E):
    for j, mrecist in enumerate(MRECIST_DISPLAY_5E):
        if (endpoint, mrecist) not in lookup_5e:
            continue
        r = lookup_5e[endpoint, mrecist]
        hr = float(r['HR_per_MBP_level_worsening'])
        lo = float(r['CI_lower'])
        hi = float(r['CI_upper'])
        p = float(r['P'])
        n = int(r['n'])
        events = int(r['events'])
        if p < 0.001:
            p_txt = 'P<0.001'
        else:
            p_txt = f'P={p:.3f}'
        text_color = 'white' if abs(log2hr_5e[i, j]) >= 0.62 * EFFECT_LIM_5E else COLOR_TEXT
        ax.text(j, i - 0.13, f'HR {hr:.2f}', ha='center', va='center', fontsize=15.9, fontweight='bold', color=text_color)
        ax.text(j, i + 0.08, f'95% CI {lo:.2f}–{hi:.2f}', ha='center', va='center', fontsize=12, color=text_color)
        ax.text(j, i + 0.27, f'{p_txt}   n={n}, events={events}', ha='center', va='center', fontsize=11.4, color=text_color)
ax.set_xticks([0, 1])
ax.set_xticklabels(['mRECIST responder', 'mRECIST nonresponder'])
ax.set_yticks([0, 1])
ax.set_yticklabels(['PFS', 'OS'])
for tick, endpoint in zip(ax.get_yticklabels(), ENDPOINT_ORDER_5E):
    tick.set_color(ENDPOINT_COLORS[endpoint])
    tick.set_fontweight('bold')
ax.tick_params(axis='both', length=0, pad=8)
for spine in ax.spines.values():
    spine.set_visible(False)
cbar = fig.colorbar(im, ax=ax, orientation='vertical', fraction=0.055, pad=0.055)
cbar.set_label('log2 HR per one-level MBP worsening', fontsize=12.6, labelpad=6)
cbar.ax.tick_params(labelsize=8.8, length=2.5)
cbar.outline.set_visible(False)
fig.subplots_adjust(left=0.19, right=0.88, top=0.82, bottom=0.11)
STEM_5E = MAIN_OUT_DIR / get_value('science.paths.directory_079', 'directory_079')
fig.savefig(str(STEM_5E) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_5E) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_5E) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
result_5e.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_100', 'result_100.csv'), index=False, encoding='utf-8-sig')



In [ ]:
N_CV_SPLITS_5F = 5
N_CV_REPEATS_5F = 5
CV_RANDOM_STATE_5F = 20260809

def fit_cox_safely_5f(train_df, duration_col, event_col, covariates):
    fit_df = train_df[[duration_col, event_col] + covariates].copy()
    cph = CoxPHFitter()
    try:
        cph.fit(fit_df, duration_col=duration_col, event_col=event_col, show_progress=False)
        penalizer_used = 0.0
    except Exception:
        cph = CoxPHFitter(penalizer=0.01)
        cph.fit(fit_df, duration_col=duration_col, event_col=event_col, show_progress=False)
        penalizer_used = 0.01
    return (cph, penalizer_used)

def cindex_from_model_5f(cph, test_df, duration_col, event_col, covariates):
    log_risk = cph.predict_log_partial_hazard(test_df[covariates]).to_numpy(dtype=float).reshape(-1)
    return float(concordance_index(test_df[duration_col].to_numpy(dtype=float), -log_risk, test_df[event_col].to_numpy(dtype=int)))

def full_data_nested_lrt_5f(data, duration_col, event_col, base_covars, extended_covars):
    d = data[[duration_col, event_col] + extended_covars].dropna().copy()
    d[event_col] = d[event_col].astype(int)
    base_model = CoxPHFitter()
    base_model.fit(d[[duration_col, event_col] + base_covars], duration_col=duration_col, event_col=event_col, show_progress=False)
    ext_model = CoxPHFitter()
    ext_model.fit(d[[duration_col, event_col] + extended_covars], duration_col=duration_col, event_col=event_col, show_progress=False)
    lr_stat = max(0.0, 2.0 * (float(ext_model.log_likelihood_) - float(base_model.log_likelihood_)))
    df_diff = len(extended_covars) - len(base_covars)
    p_lrt = float(chi2.sf(lr_stat, df_diff))
    return {'LRT_statistic': lr_stat, 'LRT_df': int(df_diff), 'LRT_P': p_lrt, 'AIC_partial_base': float(base_model.AIC_partial_), 'AIC_partial_extended': float(ext_model.AIC_partial_), 'n_full_data': int(len(d)), 'events_full_data': int(d[event_col].sum())}

def repeated_cv_nested_cox_5f(data, endpoint, context):
    duration_col = f'{endpoint}_time'
    event_col = f'{endpoint}_status'
    if context == 'Pretreatment':
        base_covars = CLINICAL_BASE_COVARS_5F.copy()
    elif context == 'Post-mRECIST':
        base_covars = CLINICAL_BASE_COVARS_5F + ['mRECIST_R']
    else:
        raise ValueError(f'Unknown context: {context}')
    extended_covars = base_covars + ['MBP_worse']
    d = data[[duration_col, event_col] + extended_covars].dropna().copy().reset_index(drop=True)
    d[event_col] = d[event_col].astype(int)
    splitter = RepeatedStratifiedKFold(n_splits=N_CV_SPLITS_5F, n_repeats=N_CV_REPEATS_5F, random_state=CV_RANDOM_STATE_5F)
    fold_records = []
    X_dummy = np.zeros((len(d), 1))
    y_strat = d[event_col].to_numpy(dtype=int)
    for fold_index, (train_idx, test_idx) in enumerate(splitter.split(X_dummy, y_strat), start=1):
        train_df = d.iloc[train_idx].copy()
        test_df = d.iloc[test_idx].copy()
        try:
            base_model, base_pen = fit_cox_safely_5f(train_df, duration_col, event_col, base_covars)
            ext_model, ext_pen = fit_cox_safely_5f(train_df, duration_col, event_col, extended_covars)
            c_base = cindex_from_model_5f(base_model, test_df, duration_col, event_col, base_covars)
            c_ext = cindex_from_model_5f(ext_model, test_df, duration_col, event_col, extended_covars)
            if np.isfinite(c_base) and np.isfinite(c_ext):
                fold_records.append({'Endpoint': endpoint, 'Context': context, 'fold_index': int(fold_index), 'n_train': int(len(train_df)), 'n_test': int(len(test_df)), 'events_test': int(test_df[event_col].sum()), 'Cindex_without_MBP': c_base, 'Cindex_with_MBP': c_ext, 'Delta_Cindex': c_ext - c_base, 'base_penalizer': base_pen, 'extended_penalizer': ext_pen})
        except Exception as e:
            print(f'[Skipped CV fold] {endpoint} | {context} | fold {fold_index}: {e}')
    fold_df = pd.DataFrame(fold_records)
    if len(fold_df) == 0:
        raise RuntimeError(f'No valid CV folds for {endpoint} / {context}.')
    lrt_info = full_data_nested_lrt_5f(d, duration_col, event_col, base_covars, extended_covars)
    summary = {'Endpoint': endpoint, 'Context': context, 'n': int(len(d)), 'events': int(d[event_col].sum()), 'n_valid_CV_folds': int(len(fold_df)), 'Cindex_without_MBP_mean': float(fold_df['Cindex_without_MBP'].mean()), 'Cindex_without_MBP_SD': float(fold_df['Cindex_without_MBP'].std(ddof=1)), 'Cindex_with_MBP_mean': float(fold_df['Cindex_with_MBP'].mean()), 'Cindex_with_MBP_SD': float(fold_df['Cindex_with_MBP'].std(ddof=1)), 'Delta_Cindex_mean': float(fold_df['Delta_Cindex'].mean()), 'Delta_Cindex_SD': float(fold_df['Delta_Cindex'].std(ddof=1)), **lrt_info}
    return (pd.DataFrame([summary]), fold_df)
summary_parts_5f = []
fold_parts_5f = []
for endpoint in ['PFS', 'OS']:
    for context in ['Pretreatment', 'Post-mRECIST']:
        s, f = repeated_cv_nested_cox_5f(clinical_model_df, endpoint=endpoint, context=context)
        summary_parts_5f.append(s)
        fold_parts_5f.append(f)
summary_5f = pd.concat(summary_parts_5f, ignore_index=True)
folds_5f = pd.concat(fold_parts_5f, ignore_index=True)



In [ ]:
row_order_5f = [('PFS', 'Pretreatment'), ('PFS', 'Post-mRECIST'), ('OS', 'Pretreatment'), ('OS', 'Post-mRECIST')]
row_labels_5f = {('PFS', 'Pretreatment'): 'PFS  |  Clinical baseline', ('PFS', 'Post-mRECIST'): 'PFS  |  Clinical + mRECIST', ('OS', 'Pretreatment'): 'OS   |  Clinical baseline', ('OS', 'Post-mRECIST'): 'OS   |  Clinical + mRECIST'}
y_positions_5f = {key: len(row_order_5f) - 1 - i for i, key in enumerate(row_order_5f)}
fig, ax = plt.subplots(figsize=(7.6, 4.15), facecolor='white')
all_cvals_5f = pd.concat([summary_5f['Cindex_without_MBP_mean'], summary_5f['Cindex_with_MBP_mean']])
xmin_5f = max(0.45, float(all_cvals_5f.min()) - 0.055)
xmax_5f = min(0.98, float(all_cvals_5f.max()) + 0.1)
ax.axvline(0.5, color='#A0A0A0', linestyle='--', linewidth=0.9, zorder=0)
for key in row_order_5f:
    endpoint, context = key
    y = y_positions_5f[key]
    row = summary_5f[(summary_5f['Endpoint'] == endpoint) & (summary_5f['Context'] == context)]
    if len(row) != 1:
        continue
    r = row.iloc[0]
    base_c = float(r['Cindex_without_MBP_mean'])
    ext_c = float(r['Cindex_with_MBP_mean'])
    delta_c = float(r['Delta_Cindex_mean'])
    p_lrt = float(r['LRT_P'])
    color = ENDPOINT_COLORS[endpoint]
    ax.axhline(y, color=COLOR_GRID, linewidth=0.8, zorder=0)
    ax.plot([base_c, ext_c], [y, y], color=color, linewidth=2.4, alpha=0.72, zorder=2)
    ax.scatter(base_c, y, s=52.5, facecolor='white', edgecolor=color, linewidth=1.6, zorder=4)
    ax.scatter(ext_c, y, s=65, facecolor=color, edgecolor='white', linewidth=0.9, zorder=5)
    ax.text(base_c, y + 0.16, f'{base_c:.3f}', ha='center', va='bottom', fontsize=11.7, color=COLOR_MUTED)
    ax.text(ext_c, y + 0.16, f'{ext_c:.3f}', ha='center', va='bottom', fontsize=11.7, color=color, fontweight='bold')
    if p_lrt < 0.001:
        p_txt = 'LRT P<0.001'
    else:
        p_txt = f'LRT P={p_lrt:.3f}'
    ax.text(xmax_5f, y, f'ΔC={delta_c:+.3f}   {p_txt}', ha='right', va='center', fontsize=12.1, color=COLOR_TEXT)
ax.set_xlim(xmin_5f, xmax_5f)
ax.set_ylim(-0.55, len(row_order_5f) - 0.45)
ax.set_yticks([y_positions_5f[key] for key in row_order_5f])
ax.set_yticklabels([row_labels_5f[key] for key in row_order_5f])
for tick, key in zip(ax.get_yticklabels(), row_order_5f):
    endpoint = key[0]
    tick.set_color(ENDPOINT_COLORS[endpoint])
ax.set_xlabel('Repeated-CV Harrell C-index')
ax.tick_params(axis='y', length=0, pad=8)
ax.grid(False)
for spine in ['top', 'right', 'left']:
    ax.spines[spine].set_visible(False)
legend_handles_5f = [Line2D([0], [0], marker='o', linestyle='none', markerfacecolor='white', markeredgecolor=COLOR_TEXT, label='Without MBP'), Line2D([0], [0], marker='o', linestyle='none', markerfacecolor=COLOR_TEXT, markeredgecolor=COLOR_TEXT, label='+ MBP')]
ax.legend(handles=legend_handles_5f, loc='upper left', frameon=False, fontsize=12.5, ncol=2, columnspacing=1.3, handletextpad=0.5)
fig.subplots_adjust(left=0.27, right=0.94, top=0.84, bottom=0.11)
STEM_5F = MAIN_OUT_DIR / get_value('science.paths.directory_080', 'directory_080')
fig.savefig(str(STEM_5F) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_5F) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_5F) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
summary_5f.to_csv(SOURCE_OUT_DIR / get_value('science.tables.clinical_019', 'clinical_019.csv'), index=False, encoding='utf-8-sig')
folds_5f.to_csv(SOURCE_OUT_DIR / get_value('science.tables.clinical_020', 'clinical_020.csv'), index=False, encoding='utf-8-sig')




In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
OUTER_N_SPLITS = 5
OUTER_N_REPEATS = 10
OUTER_RANDOM_STATE = 20260809
BOOTSTRAP_N_S13A = 1000
BOOTSTRAP_RANDOM_STATE_S13A = 20260810

def empirical_percentile_from_training_reference(train_reference, values):
    ref = pd.to_numeric(pd.Series(train_reference), errors='coerce').dropna().to_numpy(dtype=float)
    vals = pd.to_numeric(pd.Series(values), errors='coerce').to_numpy(dtype=float)
    if len(ref) == 0:
        raise ValueError('Training reference is empty.')
    ref_sorted = np.sort(ref)
    n_ref = len(ref_sorted)
    left = np.searchsorted(ref_sorted, vals, side='left')
    right = np.searchsorted(ref_sorted, vals, side='right')
    pct = (left + right + 1) / (2.0 * n_ref)
    lower_clip = 1.0 / (2.0 * n_ref)
    upper_clip = 1.0 - lower_clip
    return np.clip(pct, lower_clip, upper_clip)

def rebuild_mbp_with_training_reference(train_df, test_df):
    train = train_df.copy()
    test = test_df.copy()
    train['_Response_benefit_outer'] = empirical_percentile_from_training_reference(train['Response_pred'], train['Response_pred'])
    test['_Response_benefit_outer'] = empirical_percentile_from_training_reference(train['Response_pred'], test['Response_pred'])
    train['_PFS_benefit_outer'] = 1.0 - empirical_percentile_from_training_reference(train['PFS_risk'], train['PFS_risk'])
    test['_PFS_benefit_outer'] = 1.0 - empirical_percentile_from_training_reference(train['PFS_risk'], test['PFS_risk'])
    train['_OS_benefit_outer'] = 1.0 - empirical_percentile_from_training_reference(train['OS_risk'], train['OS_risk'])
    test['_OS_benefit_outer'] = 1.0 - empirical_percentile_from_training_reference(train['OS_risk'], test['OS_risk'])
    train['_MRU_benefit_outer'] = (train['_Response_benefit_outer'] + train['_PFS_benefit_outer'] + train['_OS_benefit_outer']) / 3.0
    test['_MRU_benefit_outer'] = (test['_Response_benefit_outer'] + test['_PFS_benefit_outer'] + test['_OS_benefit_outer']) / 3.0
    q_low = float(train['_MRU_benefit_outer'].quantile(1 / 3))
    q_high = float(train['_MRU_benefit_outer'].quantile(2 / 3))

    def score_to_mbp_worse(score):
        if score >= q_high:
            return 0.0
        if score >= q_low:
            return 1.0
        return 2.0
    train['_MBP_worse_outer'] = train['_MRU_benefit_outer'].apply(score_to_mbp_worse)
    test['_MBP_worse_outer'] = test['_MRU_benefit_outer'].apply(score_to_mbp_worse)
    return (train, test, q_low, q_high)

def standardize_continuous_from_training(train_df, test_df, continuous_covariates):
    train = train_df.copy()
    test = test_df.copy()
    for col in continuous_covariates:
        mean_val = float(train[col].mean())
        sd_val = float(train[col].std(ddof=0))
        if not np.isfinite(sd_val) or sd_val == 0:
            train[f'_z_{col}'] = 0.0
            test[f'_z_{col}'] = 0.0
        else:
            train[f'_z_{col}'] = (train[col] - mean_val) / sd_val
            test[f'_z_{col}'] = (test[col] - mean_val) / sd_val
    return (train, test)

def fit_cox_outer(train_df, duration_col, event_col, covariates):
    fit_df = train_df[[duration_col, event_col] + covariates].copy()
    cph = CoxPHFitter()
    try:
        cph.fit(fit_df, duration_col=duration_col, event_col=event_col, show_progress=False)
        penalizer_used = 0.0
    except Exception:
        cph = CoxPHFitter(penalizer=0.01)
        cph.fit(fit_df, duration_col=duration_col, event_col=event_col, show_progress=False)
        penalizer_used = 0.01
    return (cph, penalizer_used)

def predict_logrisk_outer(model, test_df, covariates):
    return model.predict_log_partial_hazard(test_df[covariates]).to_numpy(dtype=float).reshape(-1)

def cindex_from_logrisk(duration, event, logrisk):
    return float(concordance_index(np.asarray(duration, dtype=float), -np.asarray(logrisk, dtype=float), np.asarray(event, dtype=int)))
CONTINUOUS_CLINICAL_OUTER = ['ALBI_model', 'TBS_model', 'AFP_model', 'NLR_model']
BINARY_CLINICAL_OUTER = ['PVTT_model']

def run_outer_cv_sensitivity(data, endpoint, context):
    duration_col = f'{endpoint}_time'
    event_col = f'{endpoint}_status'
    required_cols = ['sample', duration_col, event_col, 'Response_pred', 'PFS_risk', 'OS_risk', 'ALBI_model', 'TBS_model', 'AFP_model', 'NLR_model', 'PVTT_model', 'mRECIST_R']
    d = data[required_cols].dropna().copy().reset_index(drop=True)
    d[event_col] = d[event_col].astype(int)
    if context == 'Pretreatment':
        binary_base = BINARY_CLINICAL_OUTER.copy()
    elif context == 'Post-mRECIST':
        binary_base = BINARY_CLINICAL_OUTER + ['mRECIST_R']
    else:
        raise ValueError(f'Unknown context: {context}')
    splitter = RepeatedStratifiedKFold(n_splits=OUTER_N_SPLITS, n_repeats=OUTER_N_REPEATS, random_state=OUTER_RANDOM_STATE)
    X_dummy = np.zeros((len(d), 1))
    y_strat = d[event_col].to_numpy(dtype=int)
    fold_records = []
    prediction_records = []
    cutoff_records = []
    for split_index, (train_idx, test_idx) in enumerate(splitter.split(X_dummy, y_strat), start=1):
        repeat_id = (split_index - 1) // OUTER_N_SPLITS + 1
        fold_in_repeat = (split_index - 1) % OUTER_N_SPLITS + 1
        train_raw = d.iloc[train_idx].copy().reset_index(drop=True)
        test_raw = d.iloc[test_idx].copy().reset_index(drop=True)
        train_fold, test_fold, q_low, q_high = rebuild_mbp_with_training_reference(train_raw, test_raw)
        train_fold, test_fold = standardize_continuous_from_training(train_fold, test_fold, CONTINUOUS_CLINICAL_OUTER)
        continuous_z = [f'_z_{c}' for c in CONTINUOUS_CLINICAL_OUTER]
        base_covars = continuous_z + binary_base
        extended_covars = base_covars + ['_MBP_worse_outer']
        try:
            base_model, base_pen = fit_cox_outer(train_fold, duration_col, event_col, base_covars)
            ext_model, ext_pen = fit_cox_outer(train_fold, duration_col, event_col, extended_covars)
            base_risk = predict_logrisk_outer(base_model, test_fold, base_covars)
            ext_risk = predict_logrisk_outer(ext_model, test_fold, extended_covars)
            c_base = cindex_from_logrisk(test_fold[duration_col], test_fold[event_col], base_risk)
            c_ext = cindex_from_logrisk(test_fold[duration_col], test_fold[event_col], ext_risk)
            fold_records.append({'Endpoint': endpoint, 'Context': context, 'repeat_id': repeat_id, 'fold_in_repeat': fold_in_repeat, 'split_index': split_index, 'n_train': len(train_fold), 'n_test': len(test_fold), 'events_test': int(test_fold[event_col].sum()), 'Cindex_without_MBP': c_base, 'Cindex_with_MBP': c_ext, 'Delta_Cindex': c_ext - c_base, 'base_penalizer': base_pen, 'extended_penalizer': ext_pen, 'training_MBP_q1_cutoff': q_low, 'training_MBP_q2_cutoff': q_high})
            cutoff_records.append({'Endpoint': endpoint, 'Context': context, 'repeat_id': repeat_id, 'fold_in_repeat': fold_in_repeat, 'training_MBP_q1_cutoff': q_low, 'training_MBP_q2_cutoff': q_high})
            for row_i in range(len(test_fold)):
                prediction_records.append({'sample': test_fold.loc[row_i, 'sample'], 'Endpoint': endpoint, 'Context': context, 'repeat_id': repeat_id, 'fold_in_repeat': fold_in_repeat, 'time': float(test_fold.loc[row_i, duration_col]), 'event': int(test_fold.loc[row_i, event_col]), 'outer_MRU_benefit_score': float(test_fold.loc[row_i, '_MRU_benefit_outer']), 'outer_MBP_worse': float(test_fold.loc[row_i, '_MBP_worse_outer']), 'logrisk_without_MBP': float(base_risk[row_i]), 'logrisk_with_MBP': float(ext_risk[row_i])})
        except Exception as e:
            print(f'[Skipped outer fold] {endpoint} | {context} | repeat {repeat_id} fold {fold_in_repeat}: {e}')
    fold_df = pd.DataFrame(fold_records)
    pred_df = pd.DataFrame(prediction_records)
    cutoff_df = pd.DataFrame(cutoff_records)
    if len(fold_df) == 0:
        raise RuntimeError(f'No valid outer-CV folds for {endpoint} / {context}.')
    patient_pred = pred_df.groupby('sample', as_index=False).agg(time=('time', 'first'), event=('event', 'first'), logrisk_without_MBP=('logrisk_without_MBP', 'mean'), logrisk_with_MBP=('logrisk_with_MBP', 'mean'), mean_outer_MRU_benefit_score=('outer_MRU_benefit_score', 'mean'), mean_outer_MBP_worse=('outer_MBP_worse', 'mean'), n_outer_predictions=('repeat_id', 'count'))
    c_base_patient = cindex_from_logrisk(patient_pred['time'], patient_pred['event'], patient_pred['logrisk_without_MBP'])
    c_ext_patient = cindex_from_logrisk(patient_pred['time'], patient_pred['event'], patient_pred['logrisk_with_MBP'])
    delta_patient = c_ext_patient - c_base_patient
    rng = np.random.default_rng(BOOTSTRAP_RANDOM_STATE_S13A + (100 if endpoint == 'OS' else 0) + (10 if context == 'Post-mRECIST' else 0))
    n_patients = len(patient_pred)
    delta_boot = []
    for _ in range(BOOTSTRAP_N_S13A):
        idx = rng.integers(0, n_patients, n_patients)
        b = patient_pred.iloc[idx].copy()
        if b['event'].nunique() < 2:
            continue
        try:
            c_b_base = cindex_from_logrisk(b['time'], b['event'], b['logrisk_without_MBP'])
            c_b_ext = cindex_from_logrisk(b['time'], b['event'], b['logrisk_with_MBP'])
            delta_boot.append(c_b_ext - c_b_base)
        except Exception:
            continue
    delta_boot = np.asarray(delta_boot, dtype=float)
    ci_low = float(np.percentile(delta_boot, 2.5))
    ci_high = float(np.percentile(delta_boot, 97.5))
    summary = pd.DataFrame([{'Endpoint': endpoint, 'Context': context, 'n_patients': n_patients, 'events': int(patient_pred['event'].sum()), 'n_valid_outer_folds': len(fold_df), 'fold_Cindex_without_MBP_mean': float(fold_df['Cindex_without_MBP'].mean()), 'fold_Cindex_with_MBP_mean': float(fold_df['Cindex_with_MBP'].mean()), 'fold_Delta_Cindex_mean': float(fold_df['Delta_Cindex'].mean()), 'fold_Delta_Cindex_SD': float(fold_df['Delta_Cindex'].std(ddof=1)), 'fold_fraction_Delta_gt_0': float((fold_df['Delta_Cindex'] > 0).mean()), 'patient_level_outer_Cindex_without_MBP': c_base_patient, 'patient_level_outer_Cindex_with_MBP': c_ext_patient, 'patient_level_outer_Delta_Cindex': delta_patient, 'Delta_Cindex_bootstrap_CI_lower': ci_low, 'Delta_Cindex_bootstrap_CI_upper': ci_high, 'n_valid_bootstrap': len(delta_boot), 'training_MBP_q1_mean': float(cutoff_df['training_MBP_q1_cutoff'].mean()), 'training_MBP_q1_SD': float(cutoff_df['training_MBP_q1_cutoff'].std(ddof=1)), 'training_MBP_q2_mean': float(cutoff_df['training_MBP_q2_cutoff'].mean()), 'training_MBP_q2_SD': float(cutoff_df['training_MBP_q2_cutoff'].std(ddof=1))}])
    bootstrap_df = pd.DataFrame({'Endpoint': endpoint, 'Context': context, 'Delta_Cindex': delta_boot})
    return (summary, fold_df, pred_df, patient_pred, cutoff_df, bootstrap_df)
summary_parts_s13a = []
fold_parts_s13a = []
pred_parts_s13a = []
patient_parts_s13a = []
cutoff_parts_s13a = []
bootstrap_parts_s13a = []
for endpoint in ['PFS', 'OS']:
    for context in ['Pretreatment', 'Post-mRECIST']:
        s, f, p, pp, c, b = run_outer_cv_sensitivity(clinical_model_df, endpoint=endpoint, context=context)
        summary_parts_s13a.append(s)
        fold_parts_s13a.append(f)
        pred_parts_s13a.append(p)
        pp = pp.copy()
        pp['Endpoint'] = endpoint
        pp['Context'] = context
        patient_parts_s13a.append(pp)
        cutoff_parts_s13a.append(c)
        bootstrap_parts_s13a.append(b)
summary_s13a = pd.concat(summary_parts_s13a, ignore_index=True)
folds_s13a = pd.concat(fold_parts_s13a, ignore_index=True)
predictions_s13a = pd.concat(pred_parts_s13a, ignore_index=True)
patient_predictions_s13a = pd.concat(patient_parts_s13a, ignore_index=True)
cutoffs_s13a = pd.concat(cutoff_parts_s13a, ignore_index=True)
bootstrap_s13a = pd.concat(bootstrap_parts_s13a, ignore_index=True)
print('Outer-CV sensitivity summary:')



In [ ]:
row_order_s13a = [('PFS', 'Pretreatment'), ('PFS', 'Post-mRECIST'), ('OS', 'Pretreatment'), ('OS', 'Post-mRECIST')]
row_labels_s13a = {('PFS', 'Pretreatment'): 'PFS  |  Clinical baseline', ('PFS', 'Post-mRECIST'): 'PFS  |  Clinical + mRECIST', ('OS', 'Pretreatment'): 'OS   |  Clinical baseline', ('OS', 'Post-mRECIST'): 'OS   |  Clinical + mRECIST'}
y_pos_s13a = {key: len(row_order_s13a) - 1 - i for i, key in enumerate(row_order_s13a)}
fig, ax = plt.subplots(figsize=(7.6, 4.25), facecolor='white')
ax.axvline(0, color='#777777', linestyle='--', linewidth=1.1, zorder=0)
rng_jitter = np.random.default_rng(20260811)
all_delta_values = []
for key in row_order_s13a:
    endpoint, context = key
    y = y_pos_s13a[key]
    fold_sub = folds_s13a[(folds_s13a['Endpoint'] == endpoint) & (folds_s13a['Context'] == context)].copy()
    summary_row = summary_s13a[(summary_s13a['Endpoint'] == endpoint) & (summary_s13a['Context'] == context)]
    if len(summary_row) != 1:
        continue
    r = summary_row.iloc[0]
    color = ENDPOINT_COLORS[endpoint]
    fold_delta = fold_sub['Delta_Cindex'].to_numpy(dtype=float)
    all_delta_values.extend(fold_delta.tolist())
    jitter = rng_jitter.normal(loc=0, scale=0.045, size=len(fold_delta))
    ax.scatter(fold_delta, y + jitter, s=16.2, facecolor=color, edgecolor='none', alpha=0.36, zorder=1)
    point = float(r['patient_level_outer_Delta_Cindex'])
    ci_low = float(r['Delta_Cindex_bootstrap_CI_lower'])
    ci_high = float(r['Delta_Cindex_bootstrap_CI_upper'])
    all_delta_values.extend([ci_low, ci_high, point])
    ax.plot([ci_low, ci_high], [y, y], color=color, linewidth=2.4, zorder=3)
    ax.plot([ci_low, ci_low], [y - 0.07, y + 0.07], color=color, linewidth=1.4, zorder=3)
    ax.plot([ci_high, ci_high], [y - 0.07, y + 0.07], color=color, linewidth=1.4, zorder=3)
    ax.scatter(point, y, s=72.5, marker='D', facecolor=color, edgecolor='white', linewidth=0.9, zorder=5)
    ax.text(ci_high + 0.006, y, f'ΔC={point:+.3f} ({ci_low:+.3f} to {ci_high:+.3f})', ha='left', va='center', fontsize=12, color=COLOR_TEXT)
all_delta_values = np.asarray(all_delta_values, dtype=float)
xmin_s13a = min(-0.08, float(np.nanmin(all_delta_values)) - 0.025)
xmax_s13a = max(0.28, float(np.nanmax(all_delta_values)) + 0.1)
ax.set_xlim(xmin_s13a, xmax_s13a)
ax.set_ylim(-0.55, len(row_order_s13a) - 0.45)
ax.set_yticks([y_pos_s13a[key] for key in row_order_s13a])
ax.set_yticklabels([row_labels_s13a[key] for key in row_order_s13a])
for tick, key in zip(ax.get_yticklabels(), row_order_s13a):
    tick.set_color(ENDPOINT_COLORS[key[0]])
ax.set_xlabel('Outer-CV Δ Harrell C-index after adding MBP')
ax.tick_params(axis='y', length=0, pad=8)
ax.grid(False)
for y in y_pos_s13a.values():
    ax.axhline(y, color=COLOR_GRID, linewidth=0.8, zorder=0)
for spine in ['top', 'right', 'left']:
    ax.spines[spine].set_visible(False)
fig.subplots_adjust(left=0.29, right=0.88, top=0.84, bottom=0.11)
STEM_S13A = SUPP_OUT_DIR / get_value('science.paths.directory_081', 'directory_081')
fig.savefig(str(STEM_S13A) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_S13A) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(STEM_S13A) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()
print(str(STEM_S13A) + '.png')


In [ ]:
summary_s13a.to_csv(SOURCE_OUT_DIR / get_value('science.tables.metrics_050', 'metrics_050.csv'), index=False, encoding='utf-8-sig')
folds_s13a.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_101', 'result_101.csv'), index=False, encoding='utf-8-sig')
predictions_s13a.to_csv(SOURCE_OUT_DIR / get_value('science.tables.predictions_025', 'predictions_025.csv'), index=False, encoding='utf-8-sig')
patient_predictions_s13a.to_csv(SOURCE_OUT_DIR / get_value('science.tables.predictions_026', 'predictions_026.csv'), index=False, encoding='utf-8-sig')
cutoffs_s13a.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_102', 'result_102.csv'), index=False, encoding='utf-8-sig')
bootstrap_s13a.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_103', 'result_103.csv'), index=False, encoding='utf-8-sig')



In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from lifelines import CoxPHFitter
from patsy import dmatrix, build_design_matrices
from scipy.stats import chi2
COLOR_TEXT = globals().get('COLOR_TEXT', '#333333')
COLOR_GRID = globals().get('COLOR_GRID', '#E6E6E6')
FIG_DPI = globals().get('FIG_DPI', 600)
ENDPOINT_COLORS = globals().get('ENDPOINT_COLORS', {'PFS': '#3B82B8', 'OS': '#7A5AA6'})
MBP_COLORS = globals().get('MBP_COLORS', {'MBP-I': '#2F78B7', 'MBP-II': '#F2B35D', 'MBP-III': '#D95F59'})
MAIN_OUT_DIR = Path(globals().get('MAIN_OUT_DIR', str(project_path('exports/figure5_main_outputs'))))
MAIN_OUT_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_OUT_DIR = Path(globals().get('SOURCE_OUT_DIR', str(project_path('exports/figure5_source_data'))))
SOURCE_OUT_DIR.mkdir(parents=True, exist_ok=True)
if 'clinical_model_df' not in globals():
    raise NameError('clinical_model_df is not available. Please run the earlier Figure 5 cells first.')
f5_df = clinical_model_df.copy()

def first_existing_col(df, candidates, required=True, purpose='column'):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f'Cannot find {purpose}. Tried: {candidates}')
    return None

def normalize_mbp_group_labels(series):
    s = pd.Series(series).copy()
    mapping = {'MBP-I': 'MBP-I', 'MBP-II': 'MBP-II', 'MBP-III': 'MBP-III', 'I': 'MBP-I', 'II': 'MBP-II', 'III': 'MBP-III', 1: 'MBP-I', 2: 'MBP-II', 3: 'MBP-III', '1': 'MBP-I', '2': 'MBP-II', '3': 'MBP-III', 0: 'MBP-I', 1.0: 'MBP-I', 2.0: 'MBP-II', 3.0: 'MBP-III'}
    out = s.map(mapping)
    missing_mask = out.isna() & s.notna()
    if missing_mask.any():
        s_str = s.astype(str).str.upper().str.strip()
        out.loc[missing_mask & s_str.str.contains('III')] = 'MBP-III'
        out.loc[missing_mask & s_str.str.contains('II')] = 'MBP-II'
        out.loc[missing_mask & s_str.str.contains('\\bI\\b', regex=True)] = 'MBP-I'
    return out

def standardize_series(series):
    x = pd.to_numeric(series, errors='coerce')
    mu = x.mean()
    sd = x.std(ddof=0)
    if pd.isna(sd) or sd == 0:
        return pd.Series(np.zeros(len(x)), index=x.index)
    return (x - mu) / sd

def binary_standardized_effect(binary_series, group_mask):
    x = pd.to_numeric(binary_series, errors='coerce')
    x = x.dropna()
    grp = x.loc[group_mask.intersection(x.index)]
    p_all = x.mean()
    n = len(grp)
    p = grp.mean()
    denom = np.sqrt(max(p_all * (1 - p_all), 1e-08))
    effect = (p - p_all) / denom
    se_p = np.sqrt(max(p * (1 - p) / max(n, 1), 1e-08))
    se_eff = se_p / denom
    lo = effect - 1.96 * se_eff
    hi = effect + 1.96 * se_eff
    return (effect, lo, hi, p, n)

def continuous_standardized_effect(z_series, group_mask):
    x = pd.to_numeric(z_series, errors='coerce')
    x = x.dropna()
    grp = x.loc[group_mask.intersection(x.index)]
    n = len(grp)
    mean_val = grp.mean()
    se = grp.std(ddof=1) / np.sqrt(n) if n > 1 else 0.0
    lo = mean_val - 1.96 * se
    hi = mean_val + 1.96 * se
    return (mean_val, lo, hi, n)


In [ ]:
score_col = first_existing_col(f5_df, ['MRU_benefit_score', 'MRU_benefit', 'benefit_score'], required=True, purpose='continuous MRU benefit score')
pfs_time_col = first_existing_col(f5_df, ['PFS_time', 'pfs_time'], required=True, purpose='PFS time')
pfs_event_col = first_existing_col(f5_df, ['PFS_status', 'pfs_status'], required=True, purpose='PFS event')
os_time_col = first_existing_col(f5_df, ['OS_time', 'os_time'], required=True, purpose='OS time')
os_event_col = first_existing_col(f5_df, ['OS_status', 'os_status'], required=True, purpose='OS event')
candidate_adjust_covariates = [c for c in ['ALBI_model', 'TBS_model', 'AFP_model', 'NLR_model', 'PVTT_model'] if c in f5_df.columns]

def nonconstant_covariates(data, covariates, variance_tol=1e-10):
    keep = []
    dropped = []
    for c in covariates:
        x = pd.to_numeric(data[c], errors='coerce').dropna()
        if len(x) == 0:
            dropped.append((c, 'all missing'))
            continue
        if x.nunique() <= 1:
            dropped.append((c, 'constant'))
            continue
        if float(x.var(ddof=0)) <= variance_tol:
            dropped.append((c, 'near-zero variance'))
            continue
        keep.append(c)
    return (keep, dropped)

def fit_spline_cox_and_predict(data, score_col, time_col, event_col, adjust_covariates, endpoint_name, spline_df=3, n_grid=160):
    use_cols = [score_col, time_col, event_col] + adjust_covariates
    d = data[use_cols].copy().dropna().reset_index(drop=True)
    d[score_col] = pd.to_numeric(d[score_col], errors='coerce')
    d[time_col] = pd.to_numeric(d[time_col], errors='coerce')
    d[event_col] = pd.to_numeric(d[event_col], errors='coerce').astype(int)
    for c in adjust_covariates:
        d[c] = pd.to_numeric(d[c], errors='coerce')
    adjust_covariates_use, dropped_covars = nonconstant_covariates(d, adjust_covariates)
    if dropped_covars:
        print(f'[{endpoint_name}] dropped non-informative adjustment covariates:', dropped_covars)
    if d[score_col].nunique() < 8:
        raise ValueError(f'{endpoint_name}: MRU_benefit_score has too few unique values for a spline analysis.')
    base_fit = d[[time_col, event_col] + adjust_covariates_use].copy()
    cph_base = CoxPHFitter()
    cph_base.fit(base_fit, duration_col=time_col, event_col=event_col, show_progress=False)
    lin_fit = d[[time_col, event_col, score_col] + adjust_covariates_use].copy()
    cph_lin = CoxPHFitter()
    cph_lin.fit(lin_fit, duration_col=time_col, event_col=event_col, show_progress=False)
    spline_formula = f"cr(x, df={spline_df}, constraints='center') - 1"
    spline_basis = dmatrix(spline_formula, {'x': d[score_col].to_numpy(dtype=float)}, return_type='dataframe')
    spline_design_info = spline_basis.design_info
    spline_basis.columns = [f'spline_{i + 1}' for i in range(spline_basis.shape[1])]
    spline_rank = int(np.linalg.matrix_rank(spline_basis.to_numpy(dtype=float)))
    if spline_rank < spline_basis.shape[1]:
        raise RuntimeError(f'{endpoint_name}: spline basis remains rank-deficient ({spline_rank}/{spline_basis.shape[1]}).')
    spline_fit = pd.concat([d[[time_col, event_col] + adjust_covariates_use].reset_index(drop=True), spline_basis.reset_index(drop=True)], axis=1)
    model_matrix_cols = adjust_covariates_use + spline_basis.columns.tolist()
    model_matrix = spline_fit[model_matrix_cols].to_numpy(dtype=float)
    full_rank = int(np.linalg.matrix_rank(model_matrix))
    if full_rank < model_matrix.shape[1]:
        raise RuntimeError(f'{endpoint_name}: full spline Cox design matrix is rank-deficient ({full_rank}/{model_matrix.shape[1]}). Inspect adjustment covariates for collinearity.')
    cph_spline = CoxPHFitter()
    cph_spline.fit(spline_fit, duration_col=time_col, event_col=event_col, show_progress=False)
    lr_overall = max(0.0, 2.0 * (float(cph_spline.log_likelihood_) - float(cph_base.log_likelihood_)))
    df_overall = len(cph_spline.params_) - len(cph_base.params_)
    p_overall = float(chi2.sf(lr_overall, df_overall))
    lr_nl = max(0.0, 2.0 * (float(cph_spline.log_likelihood_) - float(cph_lin.log_likelihood_)))
    df_nl = len(cph_spline.params_) - len(cph_lin.params_)
    p_nonlinear = float(chi2.sf(lr_nl, df_nl))
    score_values = d[score_col].to_numpy(dtype=float)
    x_grid = np.linspace(float(np.nanmin(score_values)), float(np.nanmax(score_values)), n_grid)
    x_ref = float(np.nanmedian(score_values))
    grid_basis_raw = build_design_matrices([spline_design_info], {'x': x_grid})[0]
    ref_basis_raw = build_design_matrices([spline_design_info], {'x': np.array([x_ref], dtype=float)})[0]
    grid_basis = pd.DataFrame(np.asarray(grid_basis_raw), columns=spline_basis.columns)
    ref_basis = pd.DataFrame(np.asarray(ref_basis_raw), columns=spline_basis.columns)
    ref_covars = {}
    for c in adjust_covariates_use:
        vals = d[c]
        uniq = set(vals.dropna().unique())
        if uniq.issubset({0, 1}):
            ref_covars[c] = float(vals.mean())
        else:
            ref_covars[c] = float(vals.mean())
    design_grid = pd.DataFrame(index=np.arange(len(x_grid)))
    for c in adjust_covariates_use:
        design_grid[c] = ref_covars[c]
    design_grid = pd.concat([design_grid, grid_basis.reset_index(drop=True)], axis=1)
    design_ref = pd.DataFrame(index=[0])
    for c in adjust_covariates_use:
        design_ref[c] = ref_covars[c]
    design_ref = pd.concat([design_ref, ref_basis.reset_index(drop=True)], axis=1)
    coef_order = cph_spline.params_.index.tolist()
    X_grid = design_grid[coef_order].to_numpy(dtype=float)
    X_ref = design_ref[coef_order].iloc[0].to_numpy(dtype=float)
    coef = cph_spline.params_.loc[coef_order].to_numpy(dtype=float)
    cov = cph_spline.variance_matrix_.loc[coef_order, coef_order].to_numpy(dtype=float)
    diff = X_grid - X_ref
    log_hr = diff @ coef
    var_log_hr = np.einsum('ij,jk,ik->i', diff, cov, diff)
    var_log_hr = np.clip(var_log_hr, 0, None)
    se = np.sqrt(var_log_hr)
    hr = np.exp(log_hr)
    lo = np.exp(log_hr - 1.96 * se)
    hi = np.exp(log_hr + 1.96 * se)
    plot_df = pd.DataFrame({'Endpoint': endpoint_name, 'score': x_grid, 'HR': hr, 'HR_lower': lo, 'HR_upper': hi, 'score_reference': x_ref})
    stat_row = {'Endpoint': endpoint_name, 'n': int(len(d)), 'events': int(d[event_col].sum()), 'score_reference': x_ref, 'spline_df': int(spline_df), 'spline_basis_rank': spline_rank, 'full_design_rank': full_rank, 'full_design_n_columns': int(model_matrix.shape[1]), 'P_overall': p_overall, 'P_nonlinear': p_nonlinear, 'LR_overall': lr_overall, 'LR_nonlinear': lr_nl, 'df_overall': int(df_overall), 'df_nonlinear': int(df_nl), 'adjust_covariates': ', '.join(adjust_covariates_use), 'dropped_covariates': '; '.join([f'{c}:{reason}' for c, reason in dropped_covars]), 'fit_mode': 'unpenalized_centered_natural_cubic_spline'}
    return (plot_df, pd.DataFrame([stat_row]))
g_pfs_plot, g_pfs_stats = fit_spline_cox_and_predict(f5_df, score_col=score_col, time_col=pfs_time_col, event_col=pfs_event_col, adjust_covariates=candidate_adjust_covariates, endpoint_name='PFS', spline_df=3)
g_os_plot, g_os_stats = fit_spline_cox_and_predict(f5_df, score_col=score_col, time_col=os_time_col, event_col=os_event_col, adjust_covariates=candidate_adjust_covariates, endpoint_name='OS', spline_df=3)
figure5g_plot_df = pd.concat([g_pfs_plot, g_os_plot], ignore_index=True)
figure5g_stats_df = pd.concat([g_pfs_stats, g_os_stats], ignore_index=True)
print('Figure 5G spline fit diagnostics:')



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.5), sharey=True, facecolor='white')
endpoint_order = ['PFS', 'OS']
for ax, endpoint in zip(axes, endpoint_order):
    d = figure5g_plot_df[figure5g_plot_df['Endpoint'] == endpoint].copy()
    s = figure5g_stats_df[figure5g_stats_df['Endpoint'] == endpoint].iloc[0]
    color = ENDPOINT_COLORS[endpoint]
    ax.fill_between(d['score'].to_numpy(dtype=float), d['HR_lower'].to_numpy(dtype=float), d['HR_upper'].to_numpy(dtype=float), color=color, alpha=0.42, linewidth=0, zorder=1)
    ax.plot(d['score'].to_numpy(dtype=float), d['HR'].to_numpy(dtype=float), color=color, linewidth=2.5, zorder=2)
    ax.axhline(1.0, color='#8A8A8A', linestyle='--', linewidth=1.1, zorder=0)
    x_ref = float(s['score_reference'])
    ax.axvline(x_ref, color='#B3B3B3', linestyle=':', linewidth=1.1, zorder=0)
    ax.set_xlabel('Continuous MRU benefit score', fontsize=16.1, color=COLOR_TEXT)
    ax.grid(False)
    ax.text(0.03, 0.96, f"P overall = {s['P_overall']:.3g}\nP nonlinear = {s['P_nonlinear']:.3g}", transform=ax.transAxes, ha='left', va='top', fontsize=13.4, color=COLOR_TEXT, bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor='none', alpha=0.86))
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.tick_params(axis='both', labelsize=10.7, colors=COLOR_TEXT)
axes[0].set_ylabel('Hazard ratio (vs median score)', fontsize=16.1, color=COLOR_TEXT)
fig.subplots_adjust(left=0.1, right=0.98, top=0.84, bottom=0.11, wspace=0.16)
FIG5G_STEM = MAIN_OUT_DIR / get_value('science.paths.directory_082', 'directory_082')
fig.savefig(str(FIG5G_STEM) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(FIG5G_STEM) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(FIG5G_STEM) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()



In [ ]:
mbp_col = first_existing_col(f5_df, ['MBP_raw', 'MBP_group', 'MBP', 'MBP_label', 'MBP_class'], required=True, purpose='MBP group column')
f5_df['_MBP_group_norm'] = normalize_mbp_group_labels(f5_df[mbp_col])
phenotype_specs = [{'display': 'ALBI', 'type': 'continuous', 'candidates': ['ALBI', 'ALBI_model']}, {'display': 'TBS', 'type': 'continuous', 'candidates': ['TBS', 'TBS_model']}, {'display': 'AFP', 'type': 'continuous', 'candidates': ['AFP', 'AFP_model', 'AFP_binary']}, {'display': 'NLR', 'type': 'continuous', 'candidates': ['NLR', 'NLR_model']}, {'display': 'PVTT', 'type': 'binary', 'candidates': ['PVTT', 'PVTT_model']}, {'display': 'Tumor number', 'type': 'binary', 'candidates': ['Number', 'Tumor number', 'Tumor_number']}, {'display': 'Metastasis', 'type': 'binary', 'candidates': ['Metastasis', 'metastasis']}]
resolved_specs = []
for spec in phenotype_specs:
    col = first_existing_col(f5_df, spec['candidates'], required=False, purpose=spec['display'])
    if col is not None:
        resolved_specs.append({'display': spec['display'], 'type': spec['type'], 'col': col})
if len(resolved_specs) == 0:
    raise RuntimeError('No phenotype variables found for Figure 5H.')
if any((x['display'] == 'Tumor number' for x in resolved_specs)):
    tnum_col = [x['col'] for x in resolved_specs if x['display'] == 'Tumor number'][0]
    tnum_vals = pd.to_numeric(f5_df[tnum_col], errors='coerce')
    unique_non_na = sorted(pd.unique(tnum_vals.dropna()))
    if set(unique_non_na).issubset({0, 1}):
        f5_df['_TumorNumber_binary_for_H'] = tnum_vals
    else:
        f5_df['_TumorNumber_binary_for_H'] = (tnum_vals > 1).astype(float)
    for spec in resolved_specs:
        if spec['display'] == 'Tumor number':
            spec['col'] = '_TumorNumber_binary_for_H'
            spec['type'] = 'binary'
for spec in resolved_specs:
    if spec['display'] == 'AFP':
        vals = pd.to_numeric(f5_df[spec['col']], errors='coerce')
        uniq = set(pd.unique(vals.dropna()))
        if uniq.issubset({0, 1}):
            spec['type'] = 'binary'
group_order = ['MBP-I', 'MBP-II', 'MBP-III']
group_order = [g for g in group_order if g in set(f5_df['_MBP_group_norm'].dropna().unique())]
h_rows = []
for spec in resolved_specs:
    col = spec['col']
    var_type = spec['type']
    if var_type == 'continuous':
        z = standardize_series(f5_df[col])
        for grp in group_order:
            mask = f5_df.index[f5_df['_MBP_group_norm'] == grp]
            eff, lo, hi, n = continuous_standardized_effect(z, mask)
            h_rows.append({'Variable': spec['display'], 'Variable_type': 'continuous', 'MBP_group': grp, 'Effect': eff, 'Lower': lo, 'Upper': hi, 'n': n, 'Raw_group_mean_or_rate': pd.to_numeric(f5_df.loc[mask, col], errors='coerce').mean()})
    else:
        bin_vals = pd.to_numeric(f5_df[col], errors='coerce')
        for grp in group_order:
            mask = f5_df.index[f5_df['_MBP_group_norm'] == grp]
            eff, lo, hi, p, n = binary_standardized_effect(bin_vals, mask)
            h_rows.append({'Variable': spec['display'], 'Variable_type': 'binary', 'MBP_group': grp, 'Effect': eff, 'Lower': lo, 'Upper': hi, 'n': n, 'Raw_group_mean_or_rate': p})
figure5h_df = pd.DataFrame(h_rows)
variable_order = [spec['display'] for spec in resolved_specs]



In [ ]:
y_positions = {var: len(variable_order) - 1 - i for i, var in enumerate(variable_order)}
group_offsets = {'MBP-I': -0.18, 'MBP-II': 0.0, 'MBP-III': 0.18}
fig, ax = plt.subplots(figsize=(8.5, 4.2), facecolor='white')
ax.axvline(0, color='#8A8A8A', linestyle='--', linewidth=1.1, zorder=0)
all_vals = []
for _, row in figure5h_df.iterrows():
    var = row['Variable']
    grp = row['MBP_group']
    y = y_positions[var] + group_offsets.get(grp, 0.0)
    x = float(row['Effect'])
    lo = float(row['Lower'])
    hi = float(row['Upper'])
    color = MBP_COLORS.get(grp, '#666666')
    all_vals.extend([lo, hi, x])
    ax.plot([lo, hi], [y, y], color=color, linewidth=2.1, zorder=2)
    ax.scatter(x, y, s=52.5, color=color, edgecolor='white', linewidth=0.8, zorder=3)
for var in variable_order:
    y0 = y_positions[var]
    ax.axhline(y0, color=COLOR_GRID, linewidth=0.7, zorder=0)
x_min = min(-1.8, float(np.nanmin(all_vals)) - 0.25)
x_max = max(1.8, float(np.nanmax(all_vals)) + 0.25)
ax.set_xlim(x_min, x_max)
ax.set_ylim(-0.7, len(variable_order) - 0.3)
ax.set_yticks([y_positions[v] for v in variable_order])
ax.set_yticklabels(variable_order, fontsize=15.5, color=COLOR_TEXT)
ax.set_xlabel('Standardized phenotype deviation from cohort average', fontsize=16.1, color=COLOR_TEXT)
ax.tick_params(axis='x', labelsize=10.7, colors=COLOR_TEXT)
ax.tick_params(axis='y', length=0)
for spine in ['top', 'right', 'left']:
    ax.spines[spine].set_visible(False)
ax.grid(False)
legend_elements = [Line2D([0], [0], marker='o', color=MBP_COLORS[g], markersize=7.3, linewidth=2.1, label=g) for g in group_order]
ax.legend(handles=legend_elements, loc='upper right', frameon=False, fontsize=14.4, title='MBP group', title_fontsize=15)
fig.subplots_adjust(left=0.21, right=0.97, top=0.84, bottom=0.11)
FIG5H_STEM = MAIN_OUT_DIR / get_value('science.paths.directory_083', 'directory_083')
fig.savefig(str(FIG5H_STEM) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(FIG5H_STEM) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(FIG5H_STEM) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()



In [ ]:
figure5g_plot_df.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_104', 'result_104.csv'), index=False, encoding='utf-8-sig')
figure5g_stats_df.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_105', 'result_105.csv'), index=False, encoding='utf-8-sig')
figure5h_df.to_csv(SOURCE_OUT_DIR / get_value('science.tables.clinical_021', 'clinical_021.csv'), index=False, encoding='utf-8-sig')



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.multitest import multipletests
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test
SUPP_OUT_DIR = Path(globals().get('SUPP_OUT_DIR', MAIN_OUT_DIR.parent / 'Supplementary' if 'MAIN_OUT_DIR' in globals() else str(project_path('exports/figure5_supplementary'))))
SUPP_OUT_DIR.mkdir(parents=True, exist_ok=True)
if 'clinical_model_df' not in globals():
    raise NameError('clinical_model_df is not available. Please run the earlier Figure 5 cells first.')
supp_df = clinical_model_df.copy()

def first_existing_col_supp(df, candidates, required=True, purpose='column'):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f'Cannot find {purpose}. Tried: {candidates}')
    return None
MBP_COL_SUPP = first_existing_col_supp(supp_df, ['MBP_raw', 'MBP_group', 'MBP', 'MBP_label', 'MBP_class'], required=True, purpose='MBP group')

def canonicalize_mbp_supp(series):
    s = pd.Series(series).copy()
    out = pd.Series(np.nan, index=s.index, dtype=object)
    s_str = s.astype(str).str.upper().str.strip()
    out.loc[s_str.isin(['MBP-I', 'I', '1', '1.0', '0', '0.0'])] = 'MBP-I'
    out.loc[s_str.isin(['MBP-II', 'II', '2', '2.0'])] = 'MBP-II'
    out.loc[s_str.isin(['MBP-III', 'III', '3', '3.0'])] = 'MBP-III'
    return out
supp_df['_MBP'] = canonicalize_mbp_supp(supp_df[MBP_COL_SUPP])
MBP_ORDER_SUPP = ['MBP-I', 'MBP-II', 'MBP-III']
MBP_WORSE_MAP_SUPP = {'MBP-I': 0, 'MBP-II': 1, 'MBP-III': 2}
supp_df['_MBP_worse'] = supp_df['_MBP'].map(MBP_WORSE_MAP_SUPP)
RESPONSE_COL_SUPP = first_existing_col_supp(supp_df, ['Response_observed', 'Response', 'mRECIST_R'], required=True, purpose='observed mRECIST response')
response_raw_supp = pd.to_numeric(supp_df[RESPONSE_COL_SUPP], errors='coerce')
supp_df['_mRECIST_R'] = response_raw_supp
PFS_TIME_COL_SUPP = first_existing_col_supp(supp_df, ['PFS_time', 'pfs_time'], required=True, purpose='PFS time')
PFS_STATUS_COL_SUPP = first_existing_col_supp(supp_df, ['PFS_status', 'pfs_status'], required=True, purpose='PFS status')
OS_TIME_COL_SUPP = first_existing_col_supp(supp_df, ['OS_time', 'os_time'], required=True, purpose='OS time')
OS_STATUS_COL_SUPP = first_existing_col_supp(supp_df, ['OS_status', 'os_status'], required=True, purpose='OS status')

def infer_survival_time_unit_supp(values):
    x = pd.to_numeric(pd.Series(values), errors='coerce').dropna()
    if len(x) == 0:
        return 'months'
    return 'days' if float(x.median()) > 60 else 'months'

def time_to_months_supp(values):
    x = pd.to_numeric(values, errors='coerce')
    unit = infer_survival_time_unit_supp(x)
    if unit == 'days':
        return (x / 30.4375, unit)
    return (x, unit)
supp_df['_PFS_months'], PFS_UNIT_DETECTED_SUPP = time_to_months_supp(supp_df[PFS_TIME_COL_SUPP])
supp_df['_OS_months'], OS_UNIT_DETECTED_SUPP = time_to_months_supp(supp_df[OS_TIME_COL_SUPP])
print('Detected survival time units:', {'PFS': PFS_UNIT_DETECTED_SUPP, 'OS': OS_UNIT_DETECTED_SUPP})
COLOR_TEXT = globals().get('COLOR_TEXT', '#333333')
COLOR_GRID = globals().get('COLOR_GRID', '#E6E6E6')
FIG_DPI = globals().get('FIG_DPI', 600)
MBP_COLORS = globals().get('MBP_COLORS', {'MBP-I': '#2F78B7', 'MBP-II': '#F2B35D', 'MBP-III': '#D95F59'})
ENDPOINT_COLORS = globals().get('ENDPOINT_COLORS', {'PFS': '#3B82B8', 'OS': '#7A5AA6'})


In [ ]:
response_rows_s10 = []
for grp in MBP_ORDER_SUPP:
    d = supp_df[supp_df['_MBP'] == grp][['_mRECIST_R']].dropna().copy()
    n = int(len(d))
    responders = int(d['_mRECIST_R'].sum())
    rate = responders / n if n > 0 else np.nan
    if n > 0:
        ci_low, ci_high = proportion_confint(count=responders, nobs=n, alpha=0.08, method='wilson')
    else:
        ci_low = np.nan
        ci_high = np.nan
    response_rows_s10.append({'MBP_group': grp, 'n': n, 'responders': responders, 'response_rate': rate, 'CI_lower': float(ci_low), 'CI_upper': float(ci_high)})
response_summary_s10 = pd.DataFrame(response_rows_s10)
response_table_s10 = pd.crosstab(supp_df['_MBP'], supp_df['_mRECIST_R'])
if response_table_s10.shape[0] >= 2 and response_table_s10.shape[1] >= 2:
    chi2_stat_s10, chi2_p_s10, chi2_df_s10, _ = chi2_contingency(response_table_s10)
else:
    chi2_stat_s10 = np.nan
    chi2_p_s10 = np.nan
    chi2_df_s10 = np.nan
pairwise_response_records_s10 = []
pairwise_comparisons_s10 = [('MBP-I', 'MBP-II'), ('MBP-I', 'MBP-III'), ('MBP-II', 'MBP-III')]
for better_grp, worse_grp in pairwise_comparisons_s10:
    d = supp_df[supp_df['_MBP'].isin([better_grp, worse_grp])][['_MBP', '_mRECIST_R']].dropna().copy()
    d['_NR'] = 1 - d['_mRECIST_R']
    table = pd.crosstab(d['_MBP'], d['_NR'])
    table = table.reindex(index=[worse_grp, better_grp], columns=[1, 0], fill_value=0)
    odds_ratio, p = fisher_exact(table.to_numpy(), alternative='two-sided')
    pairwise_response_records_s10.append({'Better_group': better_grp, 'Worse_group': worse_grp, 'Contrast': f'{worse_grp} vs {better_grp}', 'Adverse_outcome': 'mRECIST nonresponse', 'Odds_ratio': float(odds_ratio), 'P_raw': float(p)})
pairwise_response_s10 = pd.DataFrame(pairwise_response_records_s10)
if len(pairwise_response_s10) > 0:
    pairwise_response_s10['P_FDR'] = multipletests(pairwise_response_s10['P_raw'], method='fdr_bh')[1]

def pairwise_cox_hr_s10(data, endpoint, better_grp, worse_grp):
    if endpoint == 'PFS':
        time_col = '_PFS_months'
        status_col = PFS_STATUS_COL_SUPP
    elif endpoint == 'OS':
        time_col = '_OS_months'
        status_col = OS_STATUS_COL_SUPP
    else:
        raise ValueError(endpoint)
    d = data[data['_MBP'].isin([better_grp, worse_grp])][[time_col, status_col, '_MBP']].dropna().copy()
    d['_worse_vs_better'] = (d['_MBP'] == worse_grp).astype(int)
    d[status_col] = pd.to_numeric(d[status_col], errors='coerce').astype(int)
    fit_df = d[[time_col, status_col, '_worse_vs_better']].copy()
    cph = CoxPHFitter()
    try:
        cph.fit(fit_df, duration_col=time_col, event_col=status_col, show_progress=False)
        penalizer_used = 0.0
    except Exception:
        cph = CoxPHFitter(penalizer=0.01)
        cph.fit(fit_df, duration_col=time_col, event_col=status_col, show_progress=False)
        penalizer_used = 0.01
    s = cph.summary.loc['_worse_vs_better']
    return {'Endpoint': endpoint, 'Better_group': better_grp, 'Worse_group': worse_grp, 'Contrast': f'{worse_grp} vs {better_grp}', 'HR': float(s['exp(coef)']), 'CI_lower': float(s['exp(coef) lower 95%']), 'CI_upper': float(s['exp(coef) upper 95%']), 'P_raw': float(s['p']), 'n': int(len(d)), 'events': int(d[status_col].sum()), 'penalizer': penalizer_used}
pairwise_survival_records_s10 = []
for endpoint in ['PFS', 'OS']:
    for better_grp, worse_grp in pairwise_comparisons_s10:
        pairwise_survival_records_s10.append(pairwise_cox_hr_s10(supp_df, endpoint=endpoint, better_grp=better_grp, worse_grp=worse_grp))
pairwise_survival_s10 = pd.DataFrame(pairwise_survival_records_s10)
pairwise_survival_s10['P_FDR'] = np.nan
for endpoint in ['PFS', 'OS']:
    idx = pairwise_survival_s10['Endpoint'] == endpoint
    pairwise_survival_s10.loc[idx, 'P_FDR'] = multipletests(pairwise_survival_s10.loc[idx, 'P_raw'], method='fdr_bh')[1]




In [ ]:
fig = plt.figure(figsize=(9.4, 4.4), facecolor='white')
gs = fig.add_gridspec(1, 2, width_ratios=[0.92, 1.35], wspace=0.34)
ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])
x_positions = np.arange(len(MBP_ORDER_SUPP))
for i, grp in enumerate(MBP_ORDER_SUPP):
    row = response_summary_s10[response_summary_s10['MBP_group'] == grp]
    if len(row) != 1:
        continue
    r = row.iloc[0]
    rate = float(r['response_rate'])
    lo = float(r['CI_lower'])
    hi = float(r['CI_upper'])
    color = MBP_COLORS[grp]
    ax_a.plot([i, i], [lo, hi], color=color, linewidth=2.4, zorder=2)
    ax_a.plot([i - 0.045, i + 0.045], [lo, lo], color=color, linewidth=1.3, zorder=2)
    ax_a.plot([i - 0.045, i + 0.045], [hi, hi], color=color, linewidth=1.3, zorder=2)
    ax_a.scatter(i, rate, s=75, facecolor=color, edgecolor='white', linewidth=0.9, zorder=3)
    ax_a.text(i, min(0.99, hi + 0.055), f"{100 * rate:.1f}%\n{int(r['responders'])}/{int(r['n'])}", ha='center', va='bottom', fontsize=12.5, color=COLOR_TEXT)
ax_a.set_xticks(x_positions)
ax_a.set_xticklabels(MBP_ORDER_SUPP)
for tick, grp in zip(ax_a.get_xticklabels(), MBP_ORDER_SUPP):
    tick.set_color(MBP_COLORS[grp])
    tick.set_fontweight('bold')
ax_a.set_ylim(0, 1.08)
ax_a.set_ylabel('Observed mRECIST responder rate', fontsize=15.5)
ax_a.set_xlabel('MBP group', fontsize=15.5)
ax_a.grid(False)
ax_a.spines['top'].set_visible(False)
ax_a.spines['right'].set_visible(False)
if np.isfinite(chi2_p_s10):
    p_txt = 'P<0.001' if chi2_p_s10 < 0.001 else f'P={chi2_p_s10:.3f}'
    ax_a.text(0.02, 0.97, f'Global χ² {p_txt}', transform=ax_a.transAxes, ha='left', va='top', fontsize=12.6, color=COLOR_TEXT)
contrast_order_s10 = ['MBP-II vs MBP-I', 'MBP-III vs MBP-I', 'MBP-III vs MBP-II']
y_base = {'MBP-II vs MBP-I': 2, 'MBP-III vs MBP-I': 1, 'MBP-III vs MBP-II': 0}
endpoint_offset = {'PFS': 0.12, 'OS': -0.12}
ax_b.axvline(1.0, color='#888888', linestyle='--', linewidth=1.1, zorder=0)
for _, row in pairwise_survival_s10.iterrows():
    contrast = row['Contrast']
    endpoint = row['Endpoint']
    if contrast not in y_base:
        continue
    y = y_base[contrast] + endpoint_offset[endpoint]
    hr = float(row['HR'])
    lo = float(row['CI_lower'])
    hi = float(row['CI_upper'])
    color = ENDPOINT_COLORS[endpoint]
    ax_b.plot([lo, hi], [y, y], color=color, linewidth=2.1, zorder=2)
    ax_b.scatter(hr, y, s=47.5, facecolor=color, edgecolor='white', linewidth=0.8, zorder=3)
    p_fdr = float(row['P_FDR'])
    p_txt = 'FDR<0.001' if p_fdr < 0.001 else f'FDR={p_fdr:.3f}'
    ax_b.text(hi * 1.1, y, f'{hr:.2f} ({lo:.2f}–{hi:.2f}); {p_txt}', ha='left', va='center', fontsize=11.7, color=COLOR_TEXT)
positive_lows = pairwise_survival_s10['CI_lower'].replace([np.inf, -np.inf], np.nan).dropna()
positive_highs = pairwise_survival_s10['CI_upper'].replace([np.inf, -np.inf], np.nan).dropna()
xmin = max(0.25, float(positive_lows.min()) / 1.7)
xmax = max(8.0, float(positive_highs.max()) * 2.2)
ax_b.set_xscale('log')
ax_b.set_xlim(xmin, xmax)
ax_b.set_yticks([2, 1, 0])
ax_b.set_yticklabels(contrast_order_s10)
ax_b.set_xlabel('Hazard ratio for worse vs better MBP group (log scale)', fontsize=15.5)
ax_b.tick_params(axis='y', length=0, labelsize=10.4)
ax_b.grid(False)
for spine in ['top', 'right', 'left']:
    ax_b.spines[spine].set_visible(False)
legend_handles = [Line2D([0], [0], marker='o', color=ENDPOINT_COLORS['PFS'], markerfacecolor=ENDPOINT_COLORS['PFS'], linewidth=1.9, label='PFS'), Line2D([0], [0], marker='o', color=ENDPOINT_COLORS['OS'], markerfacecolor=ENDPOINT_COLORS['OS'], linewidth=1.9, label='OS')]
ax_b.legend(handles=legend_handles, loc='upper left', frameon=False, fontsize=13, ncol=2)
fig.subplots_adjust(left=0.09, right=0.86, top=0.84, bottom=0.11)
FIGS10_STEM = SUPP_OUT_DIR / get_value('science.paths.directory_084', 'directory_084')
fig.savefig(str(FIGS10_STEM) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(FIGS10_STEM) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(FIGS10_STEM) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
response_summary_s10.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_106', 'result_106.csv'), index=False, encoding='utf-8-sig')
pairwise_response_s10.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_107', 'result_107.csv'), index=False, encoding='utf-8-sig')
pairwise_survival_s10.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_108', 'result_108.csv'), index=False, encoding='utf-8-sig')



In [ ]:
def within_mrecist_ordinal_cox_s11(data, endpoint, mrecist_value):
    if endpoint == 'PFS':
        time_col = '_PFS_months'
        status_col = PFS_STATUS_COL_SUPP
    elif endpoint == 'OS':
        time_col = '_OS_months'
        status_col = OS_STATUS_COL_SUPP
    else:
        raise ValueError(endpoint)
    d = data[data['_mRECIST_R'] == mrecist_value][[time_col, status_col, '_MBP_worse']].dropna().copy()
    d[status_col] = pd.to_numeric(d[status_col], errors='coerce').astype(int)
    cph = CoxPHFitter()
    cph.fit(d, duration_col=time_col, event_col=status_col, show_progress=False)
    s = cph.summary.loc['_MBP_worse']
    return {'Endpoint': endpoint, 'mRECIST': 'Responder' if mrecist_value == 1 else 'Nonresponder', 'n': int(len(d)), 'events': int(d[status_col].sum()), 'HR_per_one_level_MBP_worsening': float(s['exp(coef)']), 'CI_lower': float(s['exp(coef) lower 95%']), 'CI_upper': float(s['exp(coef) upper 95%']), 'P': float(s['p'])}
s11_effect_rows = []
for endpoint in ['PFS', 'OS']:
    for mrecist_value in [1, 0]:
        s11_effect_rows.append(within_mrecist_ordinal_cox_s11(supp_df, endpoint=endpoint, mrecist_value=mrecist_value))
s11_effect_summary = pd.DataFrame(s11_effect_rows)


def draw_km_panel_s11(ax, data, endpoint, mrecist_value, panel_title):
    if endpoint == 'PFS':
        time_col = '_PFS_months'
        status_col = PFS_STATUS_COL_SUPP
    elif endpoint == 'OS':
        time_col = '_OS_months'
        status_col = OS_STATUS_COL_SUPP
    else:
        raise ValueError(endpoint)
    d = data[data['_mRECIST_R'] == mrecist_value][[time_col, status_col, '_MBP', '_MBP_worse']].dropna().copy()
    d[status_col] = pd.to_numeric(d[status_col], errors='coerce').astype(int)
    available_groups = [g for g in MBP_ORDER_SUPP if (d['_MBP'] == g).sum() > 0]
    if len(available_groups) >= 2:
        lr = multivariate_logrank_test(d[time_col], d['_MBP'], d[status_col])
        global_p = float(lr.p_value)
    else:
        global_p = np.nan
    kmf = KaplanMeierFitter()
    for grp in MBP_ORDER_SUPP:
        g = d[d['_MBP'] == grp]
        if len(g) == 0:
            continue
        kmf.fit(g[time_col], event_observed=g[status_col], label=f'{grp} (n={len(g)})')
        kmf.plot_survival_function(ax=ax, ci_show=True, ci_alpha=0.46, censor_styles={'marker': '|', 'ms': 5.9, 'mew': 0.9}, color=MBP_COLORS[grp], linewidth=2.1, alpha=0.95)
    effect_row = s11_effect_summary[(s11_effect_summary['Endpoint'] == endpoint) & (s11_effect_summary['mRECIST'] == ('Responder' if mrecist_value == 1 else 'Nonresponder'))].iloc[0]
    hr = float(effect_row['HR_per_one_level_MBP_worsening'])
    lo = float(effect_row['CI_lower'])
    hi = float(effect_row['CI_upper'])
    p_ord = float(effect_row['P'])
    p_ord_txt = 'P<0.001' if p_ord < 0.001 else f'P={p_ord:.3f}'
    if np.isfinite(global_p):
        global_txt = 'global P<0.001' if global_p < 0.001 else f'global P={global_p:.3f}'
    else:
        global_txt = 'global P=NA'
    ax.text(0.035, 0.055, f'Ordinal HR {hr:.2f} ({lo:.2f}–{hi:.2f}); {p_ord_txt}\n{global_txt}', transform=ax.transAxes, ha='left', va='bottom', fontsize=12, color=COLOR_TEXT)
    ax.set_xlabel('Time (months)', fontsize=14.4)
    ax.set_ylabel('Progression-free survival' if endpoint == 'PFS' else 'Overall survival', fontsize=14.4)
    ax.set_ylim(0, 1.04)
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    legend = ax.legend(loc='upper right', frameon=False, fontsize=11.7, handlelength=2.3)
    return d
fig, axes = plt.subplots(2, 2, figsize=(9.2, 7.0), facecolor='white')
d_s11a = draw_km_panel_s11(axes[0, 0], supp_df, endpoint='PFS', mrecist_value=1, panel_title='PFS within mRECIST responders')
d_s11b = draw_km_panel_s11(axes[0, 1], supp_df, endpoint='PFS', mrecist_value=0, panel_title='PFS within mRECIST nonresponders')
d_s11c = draw_km_panel_s11(axes[1, 0], supp_df, endpoint='OS', mrecist_value=1, panel_title='OS within mRECIST responders')
d_s11d = draw_km_panel_s11(axes[1, 1], supp_df, endpoint='OS', mrecist_value=0, panel_title='OS within mRECIST nonresponders')
for ax, label in zip(axes.flatten(), ['A', 'B', 'C', 'D']):
    ax.text(-0.13, 1.04, label, transform=ax.transAxes, ha='left', va='bottom', fontsize=17.9, fontweight='bold', color=COLOR_TEXT)
fig.subplots_adjust(left=0.09, right=0.98, top=0.9, bottom=0.09, hspace=0.28, wspace=0.2)
FIGS11_STEM = SUPP_OUT_DIR / get_value('science.paths.directory_085', 'directory_085')
fig.savefig(str(FIGS11_STEM) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(FIGS11_STEM) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(FIGS11_STEM) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
s11_effect_summary.to_csv(SOURCE_OUT_DIR / get_value('science.tables.metrics_051', 'metrics_051.csv'), index=False, encoding='utf-8-sig')
supp_df[['_MBP', '_MBP_worse', '_mRECIST_R', '_PFS_months', PFS_STATUS_COL_SUPP, '_OS_months', OS_STATUS_COL_SUPP]].to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_109', 'result_109.csv'), index=False, encoding='utf-8-sig')



In [ ]:
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

def first_existing_col_s12(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def winsorize_numeric_s12(series, lower_q=0.01, upper_q=0.99):
    x = pd.to_numeric(series, errors='coerce').copy()
    valid = x.dropna()
    if len(valid) < 5:
        return x
    lo = valid.quantile(lower_q)
    hi = valid.quantile(upper_q)
    return x.clip(lower=lo, upper=hi)

def log1p_nonnegative_s12(series):
    x = pd.to_numeric(series, errors='coerce')
    x = x.copy()
    min_val = x.min(skipna=True)
    if pd.notna(min_val) and min_val < 0:
        x = x - min_val
    return np.log1p(x)

def zscore_s12(series):
    x = pd.to_numeric(series, errors='coerce')
    mu = x.mean()
    sd = x.std(ddof=0)
    if pd.isna(sd) or sd == 0:
        return pd.Series(np.nan, index=x.index)
    return (x - mu) / sd

def encode_binary_s12(series, variable_name='binary'):
    s = pd.Series(series).copy()
    numeric = pd.to_numeric(s, errors='coerce')
    uniq_num = sorted(pd.unique(numeric.dropna()))
    if len(uniq_num) == 2:
        low, high = (uniq_num[0], uniq_num[1])
        out = numeric.map({low: 0.0, high: 1.0})
        return out
    ss = s.astype(str).str.strip().str.lower()
    positive_tokens = {'1', 'yes', 'y', 'positive', 'present', 'true', 'male', 'm'}
    negative_tokens = {'0', 'no', 'n', 'negative', 'absent', 'false', 'female', 'f'}
    out = pd.Series(np.nan, index=s.index, dtype=float)
    out.loc[ss.isin(positive_tokens)] = 1.0
    out.loc[ss.isin(negative_tokens)] = 0.0
    return out

def encode_ordinal_s12(series, mapping=None):
    s = pd.Series(series).copy()
    numeric = pd.to_numeric(s, errors='coerce')
    if numeric.notna().mean() >= 0.8:
        return numeric
    if mapping is None:
        return numeric
    ss = s.astype(str).str.strip().str.upper()
    return ss.map(mapping).astype(float)

def standardized_binary_deviation_s12(binary_series):
    x = pd.to_numeric(binary_series, errors='coerce')
    p_all = x.mean()
    denom = np.sqrt(max(p_all * (1 - p_all), 1e-08))
    return (x - p_all) / denom
S12_VARIABLE_SPECS = [{'domain': 'Liver reserve', 'label': 'ALBI', 'type': 'continuous', 'candidates': ['ALBI_score_plot', 'ALBI', 'ALBI_model'], 'transform': None}, {'domain': 'Liver reserve', 'label': 'Child-Pugh', 'type': 'ordinal', 'candidates': ['Child-Pugh', 'Child_Pugh', 'ChildPugh'], 'mapping': {'A': 0, 'B': 1, 'C': 2}}, {'domain': 'Liver injury', 'label': 'ALT', 'type': 'continuous', 'candidates': ['ALT'], 'transform': 'log1p'}, {'domain': 'Liver injury', 'label': 'AST', 'type': 'continuous', 'candidates': ['AST'], 'transform': 'log1p'}, {'domain': 'Liver injury', 'label': 'Albumin', 'type': 'continuous', 'candidates': ['ALB', 'Albumin'], 'transform': None}, {'domain': 'Liver injury', 'label': 'Bilirubin', 'type': 'continuous', 'candidates': ['TBiL', 'TBIL', 'Bilirubin'], 'transform': 'log1p'}, {'domain': 'Liver injury', 'label': 'ALP', 'type': 'continuous', 'candidates': ['ALP'], 'transform': 'log1p'}, {'domain': 'Tumor burden', 'label': 'TBS', 'type': 'continuous', 'candidates': ['TBS_plot', 'TBS', 'TBS_model'], 'transform': 'log1p'}, {'domain': 'Tumor burden', 'label': 'Tumor size', 'type': 'continuous', 'candidates': ['Size', 'Tumor size', 'Tumor_size'], 'transform': 'log1p'}, {'domain': 'Tumor burden', 'label': 'Tumor number', 'type': 'continuous', 'candidates': ['Number', 'Tumor number', 'Tumor_number'], 'transform': 'log1p'}, {'domain': 'Tumor biology', 'label': 'AFP', 'type': 'continuous', 'candidates': ['AFP_raw_model', 'AFP', 'AFP_model'], 'transform': 'log1p'}, {'domain': 'Tumor burden', 'label': 'PVTT', 'type': 'binary', 'candidates': ['PVTT_plot', 'PVTT', 'PVTT_model']}, {'domain': 'Tumor burden', 'label': 'Metastasis', 'type': 'binary', 'candidates': ['Metastasis', 'metastasis']}, {'domain': 'Inflammation', 'label': 'NLR', 'type': 'continuous', 'candidates': ['NLR_raw_model', 'NLR', 'NLR_model'], 'transform': 'log1p'}, {'domain': 'Inflammation', 'label': 'PLR', 'type': 'continuous', 'candidates': ['PLR'], 'transform': 'log1p'}, {'domain': 'Inflammation', 'label': 'SII', 'type': 'continuous', 'candidates': ['SII'], 'transform': 'log1p'}, {'domain': 'Inflammation', 'label': 'LMR', 'type': 'continuous', 'candidates': ['LMR'], 'transform': 'log1p'}, {'domain': 'Inflammation', 'label': 'LDH', 'type': 'continuous', 'candidates': ['LDH'], 'transform': 'log1p'}, {'domain': 'Nutrition', 'label': 'PNI', 'type': 'continuous', 'candidates': ['PNI'], 'transform': None}, {'domain': 'Nutrition', 'label': 'BMI', 'type': 'continuous', 'candidates': ['BMI'], 'transform': None}, {'domain': 'Hematology', 'label': 'Platelets', 'type': 'continuous', 'candidates': ['PLT', 'Platelets'], 'transform': 'log1p'}, {'domain': 'Hematology', 'label': 'WBC', 'type': 'continuous', 'candidates': ['WBC'], 'transform': 'log1p'}, {'domain': 'Hematology', 'label': 'Hemoglobin', 'type': 'continuous', 'candidates': ['HGB', 'Hemoglobin'], 'transform': None}, {'domain': 'Hematology', 'label': 'FIB-4', 'type': 'continuous', 'candidates': ['FIB-4', 'FIB4'], 'transform': 'log1p'}]
s12_resolved_specs = []
for spec in S12_VARIABLE_SPECS:
    col = first_existing_col_s12(supp_df, spec['candidates'])
    if col is None:
        continue
    resolved = spec.copy()
    resolved['column'] = col
    s12_resolved_specs.append(resolved)
print(f'Figure S12 resolved {len(s12_resolved_specs)} baseline variables.')
s12_effect_rows = []
s12_test_rows = []
for spec in s12_resolved_specs:
    col = spec['column']
    label = spec['label']
    domain = spec['domain']
    var_type = spec['type']
    if var_type == 'continuous':
        x = winsorize_numeric_s12(supp_df[col])
        if spec.get('transform') == 'log1p':
            x = log1p_nonnegative_s12(x)
        x_std = zscore_s12(x)
        prepared = x_std
    elif var_type == 'ordinal':
        x = encode_ordinal_s12(supp_df[col], mapping=spec.get('mapping'))
        x = winsorize_numeric_s12(x)
        prepared = zscore_s12(x)
    elif var_type == 'binary':
        x_bin = encode_binary_s12(supp_df[col], variable_name=label)
        prepared = standardized_binary_deviation_s12(x_bin)
    else:
        continue
    for grp in MBP_ORDER_SUPP:
        mask = supp_df['_MBP'] == grp
        vals = prepared.loc[mask].dropna()
        if len(vals) == 0:
            effect = np.nan
            n = 0
        else:
            effect = float(vals.mean())
            n = int(len(vals))
        s12_effect_rows.append({'Domain': domain, 'Variable': label, 'Source_column': col, 'Variable_type': var_type, 'MBP_group': grp, 'Standardized_deviation': effect, 'n': n})
    if var_type in ['continuous', 'ordinal']:
        groups = []
        for grp in MBP_ORDER_SUPP:
            vals = prepared.loc[supp_df['_MBP'] == grp].dropna()
            if len(vals) > 0:
                groups.append(vals.to_numpy(dtype=float))
        if len(groups) >= 2:
            stat, p = kruskal(*groups)
        else:
            stat = np.nan
            p = np.nan
    else:
        original_binary = encode_binary_s12(supp_df[col], variable_name=label)
        temp = pd.DataFrame({'MBP': supp_df['_MBP'], 'Binary': original_binary}).dropna()
        table = pd.crosstab(temp['MBP'], temp['Binary'])
        if table.shape[0] >= 2 and table.shape[1] >= 2:
            stat, p, _, _ = chi2_contingency(table)
        else:
            stat = np.nan
            p = np.nan
    s12_test_rows.append({'Domain': domain, 'Variable': label, 'Source_column': col, 'Variable_type': var_type, 'Global_statistic': stat, 'P_raw': p})
s12_effect_df = pd.DataFrame(s12_effect_rows)
s12_test_df = pd.DataFrame(s12_test_rows)
valid_p_mask_s12 = s12_test_df['P_raw'].notna()
s12_test_df['P_FDR'] = np.nan
if valid_p_mask_s12.sum() > 0:
    s12_test_df.loc[valid_p_mask_s12, 'P_FDR'] = multipletests(s12_test_df.loc[valid_p_mask_s12, 'P_raw'], method='fdr_bh')[1]



In [ ]:
s12_variable_order = [spec['label'] for spec in s12_resolved_specs]
s12_domain_lookup = {spec['label']: spec['domain'] for spec in s12_resolved_specs}
effect_matrix_s12 = np.full((len(s12_variable_order), len(MBP_ORDER_SUPP)), np.nan, dtype=float)
for i, var in enumerate(s12_variable_order):
    for j, grp in enumerate(MBP_ORDER_SUPP):
        row = s12_effect_df[(s12_effect_df['Variable'] == var) & (s12_effect_df['MBP_group'] == grp)]
        if len(row) == 1:
            effect_matrix_s12[i, j] = float(row.iloc[0]['Standardized_deviation'])
max_abs_s12 = float(np.nanmax(np.abs(effect_matrix_s12)))
effect_lim_s12 = max(1.0, min(2.0, np.ceil(max_abs_s12 * 2) / 2))
S12_CMAP = LinearSegmentedColormap.from_list('s12_clinical_profile', ['#2F78B7', '#BFD6E7', '#F7F7F7', '#F1B8AD', '#D95F59'], N=256)
norm_s12 = TwoSlopeNorm(vmin=-effect_lim_s12, vcenter=0, vmax=effect_lim_s12)
fig = plt.figure(figsize=(8.8, max(6.8, 0.34 * len(s12_variable_order) + 1.9)), facecolor='white')
gs = fig.add_gridspec(1, 2, width_ratios=[4.7, 1.35], wspace=0.13)
ax_heat = fig.add_subplot(gs[0, 0])
ax_fdr = fig.add_subplot(gs[0, 1])
im = ax_heat.imshow(effect_matrix_s12, cmap=S12_CMAP, norm=norm_s12, aspect='auto', interpolation='nearest')
ax_heat.set_xticks(np.arange(-0.5, len(MBP_ORDER_SUPP), 1), minor=True)
ax_heat.set_yticks(np.arange(-0.5, len(s12_variable_order), 1), minor=True)
ax_heat.grid(which='minor', color='white', linewidth=1.8)
ax_heat.tick_params(which='minor', bottom=False, left=False)
ax_heat.set_xticks(np.arange(len(MBP_ORDER_SUPP)))
ax_heat.set_xticklabels(MBP_ORDER_SUPP, fontsize=14.7)
for tick, grp in zip(ax_heat.get_xticklabels(), MBP_ORDER_SUPP):
    tick.set_color(MBP_COLORS[grp])
    tick.set_fontweight('bold')
ax_heat.set_yticks(np.arange(len(s12_variable_order)))
ax_heat.set_yticklabels(s12_variable_order, fontsize=13.5)
ax_heat.tick_params(axis='both', length=0, pad=6)
for spine in ax_heat.spines.values():
    spine.set_visible(False)
domain_blocks_s12 = []
current_domain = None
start_idx = 0
for i, var in enumerate(s12_variable_order):
    domain = s12_domain_lookup[var]
    if current_domain is None:
        current_domain = domain
        start_idx = i
    elif domain != current_domain:
        domain_blocks_s12.append((current_domain, start_idx, i - 1))
        current_domain = domain
        start_idx = i
if current_domain is not None:
    domain_blocks_s12.append((current_domain, start_idx, len(s12_variable_order) - 1))
for domain, start, end in domain_blocks_s12:
    if start > 0:
        ax_heat.axhline(start - 0.5, color='#AFAFAF', linewidth=0.9)
    mid = (start + end) / 2
    ax_heat.text(-0.72, mid, domain, ha='right', va='center', fontsize=12.6, fontweight='bold', color='#666666', clip_on=False)
cbar = fig.colorbar(im, ax=ax_heat, orientation='horizontal', fraction=0.035, pad=0.05, aspect=30)
cbar.set_label('Standardized deviation from cohort average', fontsize=12.5)
cbar.ax.tick_params(labelsize=8.5, length=2.2)
cbar.outline.set_visible(False)
s12_test_lookup = s12_test_df.set_index('Variable')
fdr_values = []
for var in s12_variable_order:
    if var in s12_test_lookup.index:
        q = s12_test_lookup.loc[var, 'P_FDR']
        if pd.isna(q):
            fdr_values.append(np.nan)
        else:
            fdr_values.append(-np.log10(max(float(q), 1e-12)))
    else:
        fdr_values.append(np.nan)
yvals = np.arange(len(s12_variable_order))
for y, val, var in zip(yvals, fdr_values, s12_variable_order):
    if not np.isfinite(val):
        continue
    ax_fdr.plot([0, val], [y, y], color='#B8B8B8', linewidth=1.2, zorder=1)
    q = float(s12_test_lookup.loc[var, 'P_FDR'])
    point_color = '#333333' if q < 0.05 else '#B8B8B8'
    ax_fdr.scatter(val, y, s=26 if q < 0.05 else 16, facecolor=point_color, edgecolor='white', linewidth=0.6, zorder=2)
ax_fdr.axvline(-np.log10(0.05), color='#777777', linestyle='--', linewidth=0.9)
xmax_fdr = max(2.0, np.nanmax(fdr_values) + 0.5)
ax_fdr.set_xlim(0, xmax_fdr)
ax_fdr.set_ylim(len(s12_variable_order) - 0.5, -0.5)
ax_fdr.set_yticks([])
ax_fdr.set_xlabel('-log10(FDR)', fontsize=12.9)
ax_fdr.grid(False)
for spine in ['top', 'right', 'left']:
    ax_fdr.spines[spine].set_visible(False)
fig.subplots_adjust(left=0.28, right=0.96, top=0.91, bottom=0.1)
FIGS12_STEM = SUPP_OUT_DIR / get_value('science.paths.directory_086', 'directory_086')
fig.savefig(str(FIGS12_STEM) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(FIGS12_STEM) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(FIGS12_STEM) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
s12_effect_df.to_csv(SOURCE_OUT_DIR / get_value('science.tables.features_026', 'features_026.csv'), index=False, encoding='utf-8-sig')
s12_test_df.to_csv(SOURCE_OUT_DIR / get_value('science.tables.clinical_022', 'clinical_022.csv'), index=False, encoding='utf-8-sig')



In [ ]:
required_s13b = ['MBP_worse', 'ALBI_z', 'TBS_z', 'AFP_z', 'NLR_z', 'PVTT_model', 'PFS_time', 'PFS_status', 'OS_time', 'OS_status']
missing_s13b = [c for c in required_s13b if c not in clinical_model_df.columns]
if missing_s13b:
    raise KeyError(f'Run the Figure 5D shared clinical-modeling setup first. Missing: {missing_s13b}')
S13B_COVARIATES = ['MBP_worse', 'ALBI_z', 'TBS_z', 'AFP_z', 'NLR_z', 'PVTT_model']
S13B_LABELS = {'MBP_worse': 'MBP (per level worse)', 'ALBI_z': 'ALBI (per SD)', 'TBS_z': 'TBS (per SD)', 'AFP_z': 'AFP (per SD)', 'NLR_z': 'NLR (per SD)', 'PVTT_model': 'PVTT (yes vs no)'}

def fit_full_adjusted_cox_s13b(data, endpoint):
    time_col = f'{endpoint}_time'
    status_col = f'{endpoint}_status'
    d = data[[time_col, status_col] + S13B_COVARIATES].dropna().copy()
    d[status_col] = pd.to_numeric(d[status_col], errors='coerce').astype(int)
    cph = CoxPHFitter()
    try:
        cph.fit(d, duration_col=time_col, event_col=status_col, show_progress=False)
        penalizer_used = 0.0
    except Exception:
        cph = CoxPHFitter(penalizer=0.01)
        cph.fit(d, duration_col=time_col, event_col=status_col, show_progress=False)
        penalizer_used = 0.01
    rows = []
    for cov in S13B_COVARIATES:
        s = cph.summary.loc[cov]
        rows.append({'Endpoint': endpoint, 'Covariate': cov, 'Label': S13B_LABELS[cov], 'HR': float(s['exp(coef)']), 'CI_lower': float(s['exp(coef) lower 95%']), 'CI_upper': float(s['exp(coef) upper 95%']), 'P': float(s['p']), 'n': int(len(d)), 'events': int(d[status_col].sum()), 'penalizer': penalizer_used})
    return pd.DataFrame(rows)
s13b_results = pd.concat([fit_full_adjusted_cox_s13b(clinical_model_df, 'PFS'), fit_full_adjusted_cox_s13b(clinical_model_df, 'OS')], ignore_index=True)



In [ ]:
cov_order_s13b = ['MBP_worse', 'ALBI_z', 'TBS_z', 'AFP_z', 'NLR_z', 'PVTT_model']
y_base_s13b = {cov: len(cov_order_s13b) - 1 - i for i, cov in enumerate(cov_order_s13b)}
endpoint_offsets_s13b = {'PFS': 0.12, 'OS': -0.12}
fig, ax = plt.subplots(figsize=(10.9, 4.8), facecolor='white')
ax.axvline(1.0, color='#888888', linestyle='--', linewidth=1.1, zorder=0)
for _, row in s13b_results.iterrows():
    endpoint = row['Endpoint']
    cov = row['Covariate']
    y = y_base_s13b[cov] + endpoint_offsets_s13b[endpoint]
    hr = float(row['HR'])
    lo = float(row['CI_lower'])
    hi = float(row['CI_upper'])
    p = float(row['P'])
    color = ENDPOINT_COLORS[endpoint]
    ax.plot([lo, hi], [y, y], color=color, linewidth=2.1, zorder=2)
    ax.scatter(hr, y, s=47.5, facecolor=color, edgecolor='white', linewidth=0.8, zorder=3)
    p_txt = 'P<0.001' if p < 0.001 else f'P={p:.3f}'
    ax.text(hi * 1.08, y, f'{hr:.2f} ({lo:.2f}–{hi:.2f}); {p_txt}', ha='left', va='center', fontsize=11.6, color=COLOR_TEXT)
for cov in cov_order_s13b:
    ax.axhline(y_base_s13b[cov], color=COLOR_GRID, linewidth=0.6, zorder=0)
lo_all = s13b_results['CI_lower'].replace([np.inf, -np.inf], np.nan).dropna()
hi_all = s13b_results['CI_upper'].replace([np.inf, -np.inf], np.nan).dropna()
xmin_s13b = max(0.25, float(lo_all.min()) / 1.7)
xmax_s13b = max(4.0, float(hi_all.max()) * 2.2)
ax.set_xscale('log')
ax.set_xlim(xmin_s13b, xmax_s13b)
ax.set_yticks([y_base_s13b[cov] for cov in cov_order_s13b])
ax.set_yticklabels([S13B_LABELS[cov] for cov in cov_order_s13b])
ax.set_xlabel('Adjusted hazard ratio (log scale)', fontsize=15.5)
ax.tick_params(axis='y', length=0, labelsize=10.4, pad=7)
ax.grid(False)
for spine in ['top', 'right', 'left']:
    ax.spines[spine].set_visible(False)
ax.legend(handles=[Line2D([0], [0], marker='o', color=ENDPOINT_COLORS['PFS'], markerfacecolor=ENDPOINT_COLORS['PFS'], linewidth=1.9, label='PFS'), Line2D([0], [0], marker='o', color=ENDPOINT_COLORS['OS'], markerfacecolor=ENDPOINT_COLORS['OS'], linewidth=1.9, label='OS')], loc='upper left', frameon=False, fontsize=12.6, ncol=2)
fig.subplots_adjust(left=0.24, right=0.77, top=0.84, bottom=0.1)
FIGS13B_STEM = SUPP_OUT_DIR / get_value('science.paths.directory_087', 'directory_087')
fig.savefig(str(FIGS13B_STEM) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(FIGS13B_STEM) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(FIGS13B_STEM) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
if 'folds_5f' not in globals():
    raise NameError('folds_5f is not available. Run the Figure 5F repeated-CV nested-model cells first.')
ecdf_specs_s13c = [{'Endpoint': 'PFS', 'Context': 'Pretreatment', 'Label': 'PFS | Clinical baseline', 'color': ENDPOINT_COLORS['PFS'], 'linestyle': '-'}, {'Endpoint': 'PFS', 'Context': 'Post-mRECIST', 'Label': 'PFS | Clinical + mRECIST', 'color': ENDPOINT_COLORS['PFS'], 'linestyle': '--'}, {'Endpoint': 'OS', 'Context': 'Pretreatment', 'Label': 'OS | Clinical baseline', 'color': ENDPOINT_COLORS['OS'], 'linestyle': '-'}, {'Endpoint': 'OS', 'Context': 'Post-mRECIST', 'Label': 'OS | Clinical + mRECIST', 'color': ENDPOINT_COLORS['OS'], 'linestyle': '--'}]
fig, ax = plt.subplots(figsize=(7.4, 4.5), facecolor='white')
ax.axvline(0, color='#777777', linestyle=':', linewidth=1.2, zorder=0)
ecdf_summary_s13c = []
for spec in ecdf_specs_s13c:
    d = folds_5f[(folds_5f['Endpoint'] == spec['Endpoint']) & (folds_5f['Context'] == spec['Context'])].copy()
    vals = np.sort(d['Delta_Cindex'].dropna().to_numpy(dtype=float))
    if len(vals) == 0:
        continue
    y = np.arange(1, len(vals) + 1) / len(vals)
    ax.step(vals, y, where='post', color=spec['color'], linestyle=spec['linestyle'], linewidth=2.1, label=spec['Label'])
    ecdf_summary_s13c.append({'Endpoint': spec['Endpoint'], 'Context': spec['Context'], 'n_folds': int(len(vals)), 'mean_Delta_Cindex': float(np.mean(vals)), 'median_Delta_Cindex': float(np.median(vals)), 'fraction_Delta_gt_0': float(np.mean(vals > 0))})
ax.set_xlabel('Fold-level Δ Harrell C-index after adding MBP', fontsize=15.5)
ax.set_ylabel('Empirical cumulative fraction of CV folds', fontsize=15.5)
ax.set_ylim(0, 1.02)
all_delta_s13c = folds_5f['Delta_Cindex'].dropna().to_numpy(dtype=float)
ax.set_xlim(min(-0.08, float(np.min(all_delta_s13c)) - 0.02), max(0.3, float(np.max(all_delta_s13c)) + 0.03))
ax.grid(False)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.legend(loc='lower right', frameon=False, fontsize=12.3)
fig.subplots_adjust(left=0.12, right=0.98, top=0.84, bottom=0.11)
FIGS13C_STEM = SUPP_OUT_DIR / get_value('science.paths.directory_088', 'directory_088')
fig.savefig(str(FIGS13C_STEM) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(FIGS13C_STEM) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(FIGS13C_STEM) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()
ecdf_summary_s13c = pd.DataFrame(ecdf_summary_s13c)



In [ ]:
if 'cutoffs_s13a' not in globals():
    raise NameError('cutoffs_s13a is not available. Run the S13A outer-CV sensitivity cells first.')
cutoff_plot_df_s13d = cutoffs_s13a.copy()
dedup_cols_s13d = ['Endpoint', 'repeat_id', 'fold_in_repeat', 'training_MBP_q1_cutoff', 'training_MBP_q2_cutoff']
cutoff_plot_df_s13d = cutoff_plot_df_s13d[dedup_cols_s13d].drop_duplicates().reset_index(drop=True)
cutoff_long_s13d = pd.concat([cutoff_plot_df_s13d[['Endpoint', 'repeat_id', 'fold_in_repeat', 'training_MBP_q1_cutoff']].rename(columns={'training_MBP_q1_cutoff': 'Cutoff'}).assign(Tertile_boundary='Lower tertile boundary'), cutoff_plot_df_s13d[['Endpoint', 'repeat_id', 'fold_in_repeat', 'training_MBP_q2_cutoff']].rename(columns={'training_MBP_q2_cutoff': 'Cutoff'}).assign(Tertile_boundary='Upper tertile boundary')], ignore_index=True)
fig, ax = plt.subplots(figsize=(7.2, 4.3), facecolor='white')
positions_s13d = {('PFS', 'Lower tertile boundary'): 0.85, ('PFS', 'Upper tertile boundary'): 1.15, ('OS', 'Lower tertile boundary'): 1.85, ('OS', 'Upper tertile boundary'): 2.15}
rng_s13d = np.random.default_rng(20260812)
cutoff_summary_s13d = []
for (endpoint, boundary), pos in positions_s13d.items():
    vals = cutoff_long_s13d[(cutoff_long_s13d['Endpoint'] == endpoint) & (cutoff_long_s13d['Tertile_boundary'] == boundary)]['Cutoff'].dropna().to_numpy(dtype=float)
    if len(vals) == 0:
        continue
    color = ENDPOINT_COLORS[endpoint]
    bp = ax.boxplot([vals], positions=[pos], widths=0.18, vert=True, patch_artist=False, showfliers=False, medianprops={'color': color, 'linewidth': 2}, boxprops={'color': color, 'linewidth': 1.4}, whiskerprops={'color': color, 'linewidth': 1.2}, capprops={'color': color, 'linewidth': 1.2})
    jitter = rng_s13d.normal(loc=0, scale=0.025, size=len(vals))
    ax.scatter(np.full(len(vals), pos) + jitter, vals, s=16.2, facecolor=color, edgecolor='none', alpha=0.41, zorder=2)
    cutoff_summary_s13d.append({'Endpoint': endpoint, 'Boundary': boundary, 'n_outer_training_folds': int(len(vals)), 'Mean_cutoff': float(np.mean(vals)), 'SD_cutoff': float(np.std(vals, ddof=1)), 'Min_cutoff': float(np.min(vals)), 'Max_cutoff': float(np.max(vals))})
ax.set_xticks([1.0, 2.0])
ax.set_xticklabels(['PFS outer splits', 'OS outer splits'])
ax.set_ylabel('Training-fold MRU benefit-score cutoff', fontsize=15.5)
ax.set_xlim(0.55, 2.45)
ax.grid(False)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
legend_handles_s13d = [Line2D([0], [0], marker='s', linestyle='none', markerfacecolor='white', markeredgecolor='#666666', label='Lower tertile boundary'), Line2D([0], [0], marker='s', linestyle='none', markerfacecolor='#666666', markeredgecolor='#666666', label='Upper tertile boundary')]
for x_center in [1.0, 2.0]:
    ax.text(x_center - 0.15, ax.get_ylim()[1], 'Q1', ha='center', va='top', fontsize=12, color='#666666')
    ax.text(x_center + 0.15, ax.get_ylim()[1], 'Q2', ha='center', va='top', fontsize=12, color='#666666')
fig.subplots_adjust(left=0.13, right=0.98, top=0.84, bottom=0.11)
FIGS13D_STEM = SUPP_OUT_DIR / get_value('science.paths.directory_089', 'directory_089')
fig.savefig(str(FIGS13D_STEM) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(FIGS13D_STEM) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(FIGS13D_STEM) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()
cutoff_summary_s13d = pd.DataFrame(cutoff_summary_s13d)



In [ ]:
s13b_results.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_110', 'result_110.csv'), index=False, encoding='utf-8-sig')
ecdf_summary_s13c.to_csv(SOURCE_OUT_DIR / get_value('science.tables.metrics_052', 'metrics_052.csv'), index=False, encoding='utf-8-sig')
folds_5f[['Endpoint', 'Context', 'fold_index', 'Cindex_without_MBP', 'Cindex_with_MBP', 'Delta_Cindex']].to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_111', 'result_111.csv'), index=False, encoding='utf-8-sig')
cutoff_long_s13d.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_112', 'result_112.csv'), index=False, encoding='utf-8-sig')
cutoff_summary_s13d.to_csv(SOURCE_OUT_DIR / get_value('science.tables.metrics_053', 'metrics_053.csv'), index=False, encoding='utf-8-sig')



In [ ]:
from sklearn.metrics import cohen_kappa_score
import statsmodels.api as sm
from lifelines import CoxPHFitter
required_mru_cols_s14 = ['sample', 'Response_benefit', 'PFS_benefit', 'OS_benefit', 'MBP_raw']
missing_mru_cols_s14 = [c for c in required_mru_cols_s14 if c not in mru_df.columns]
if missing_mru_cols_s14:
    raise KeyError(f'Run the primary MBP reconstruction cells first. Missing from mru_df: {missing_mru_cols_s14}')

def assign_tertile_mbp_s14(score_series):
    score = pd.to_numeric(score_series, errors='coerce')
    q1 = float(score.quantile(1 / 3))
    q2 = float(score.quantile(2 / 3))
    group = pd.Series(np.nan, index=score.index, dtype=object)
    group.loc[score >= q2] = 'MBP-I'
    group.loc[(score >= q1) & (score < q2)] = 'MBP-II'
    group.loc[score < q1] = 'MBP-III'
    worse = group.map({'MBP-I': 0.0, 'MBP-II': 1.0, 'MBP-III': 2.0})
    return (group, worse, q1, q2)
s14_mru = mru_df[['sample', 'Response_benefit', 'PFS_benefit', 'OS_benefit', 'MBP_raw']].copy()
s14_mru['Primary_MBP'] = s14_mru['MBP_raw']
s14_mru['Primary_MBP_worse'] = s14_mru['Primary_MBP'].map({'MBP-I': 0.0, 'MBP-II': 1.0, 'MBP-III': 2.0})
s14_mru['Response_excluded_score'] = (s14_mru['PFS_benefit'] + s14_mru['OS_benefit']) / 2.0
s14_mru['Response_excluded_MBP'], s14_mru['Response_excluded_worse'], response_excluded_q1_s14, response_excluded_q2_s14 = assign_tertile_mbp_s14(s14_mru['Response_excluded_score'])
s14_mru['PFS_excluded_score'] = (s14_mru['Response_benefit'] + s14_mru['OS_benefit']) / 2.0
s14_mru['PFS_excluded_MBP'], s14_mru['PFS_excluded_worse'], pfs_excluded_q1_s14, pfs_excluded_q2_s14 = assign_tertile_mbp_s14(s14_mru['PFS_excluded_score'])
s14_mru['OS_excluded_score'] = (s14_mru['Response_benefit'] + s14_mru['PFS_benefit']) / 2.0
s14_mru['OS_excluded_MBP'], s14_mru['OS_excluded_worse'], os_excluded_q1_s14, os_excluded_q2_s14 = assign_tertile_mbp_s14(s14_mru['OS_excluded_score'])
s14_analysis_df = clinical_model_df.merge(s14_mru[['sample', 'Primary_MBP', 'Primary_MBP_worse', 'Response_excluded_score', 'Response_excluded_MBP', 'Response_excluded_worse', 'PFS_excluded_score', 'PFS_excluded_MBP', 'PFS_excluded_worse', 'OS_excluded_score', 'OS_excluded_MBP', 'OS_excluded_worse']], on='sample', how='left')
if 'Nonresponse' not in s14_analysis_df.columns:
    s14_analysis_df['Nonresponse'] = 1 - pd.to_numeric(s14_analysis_df['mRECIST_R'], errors='coerce')
S14_ADJUSTERS = [c for c in ['ALBI_z', 'TBS_z', 'AFP_z', 'NLR_z', 'PVTT_model'] if c in s14_analysis_df.columns]

def fit_response_effect_s14(data, predictor, definition_label):
    cols = ['Nonresponse', predictor] + S14_ADJUSTERS
    d = data[cols].dropna().copy()
    y = d['Nonresponse'].astype(int)
    X = sm.add_constant(d[[predictor] + S14_ADJUSTERS], has_constant='add')
    model = sm.GLM(y, X, family=sm.families.Binomial()).fit()
    beta = float(model.params[predictor])
    se = float(model.bse[predictor])
    return {'Endpoint': 'Response', 'Definition': definition_label, 'Measure': 'OR', 'Estimate': float(np.exp(beta)), 'CI_lower': float(np.exp(beta - 1.96 * se)), 'CI_upper': float(np.exp(beta + 1.96 * se)), 'P': float(model.pvalues[predictor]), 'n': int(len(d)), 'events': int(y.sum())}

def fit_survival_effect_s14(data, endpoint, predictor, definition_label):
    time_col = f'{endpoint}_time'
    status_col = f'{endpoint}_status'
    cols = [time_col, status_col, predictor] + S14_ADJUSTERS
    d = data[cols].dropna().copy()
    d[status_col] = pd.to_numeric(d[status_col], errors='coerce').astype(int)
    cph = CoxPHFitter()
    try:
        cph.fit(d, duration_col=time_col, event_col=status_col, show_progress=False)
        penalizer_used = 0.0
    except Exception:
        cph = CoxPHFitter(penalizer=0.01)
        cph.fit(d, duration_col=time_col, event_col=status_col, show_progress=False)
        penalizer_used = 0.01
    s = cph.summary.loc[predictor]
    return {'Endpoint': endpoint, 'Definition': definition_label, 'Measure': 'HR', 'Estimate': float(s['exp(coef)']), 'CI_lower': float(s['exp(coef) lower 95%']), 'CI_upper': float(s['exp(coef) upper 95%']), 'P': float(s['p']), 'n': int(len(d)), 'events': int(d[status_col].sum()), 'penalizer': penalizer_used}
s14_effect_records = []
s14_effect_records.append(fit_response_effect_s14(s14_analysis_df, predictor='Primary_MBP_worse', definition_label='Primary MBP'))
s14_effect_records.append(fit_response_effect_s14(s14_analysis_df, predictor='Response_excluded_worse', definition_label='Response-excluded MBP'))
s14_effect_records.append(fit_survival_effect_s14(s14_analysis_df, endpoint='PFS', predictor='Primary_MBP_worse', definition_label='Primary MBP'))
s14_effect_records.append(fit_survival_effect_s14(s14_analysis_df, endpoint='PFS', predictor='PFS_excluded_worse', definition_label='PFS-excluded MBP'))
s14_effect_records.append(fit_survival_effect_s14(s14_analysis_df, endpoint='OS', predictor='Primary_MBP_worse', definition_label='Primary MBP'))
s14_effect_records.append(fit_survival_effect_s14(s14_analysis_df, endpoint='OS', predictor='OS_excluded_worse', definition_label='OS-excluded MBP'))
s14_effect_df = pd.DataFrame(s14_effect_records)
S14_ENDPOINT_DEFINITIONS = {'Response': 'Response_excluded_MBP', 'PFS': 'PFS_excluded_MBP', 'OS': 'OS_excluded_MBP'}
MBP_NUMERIC_S14 = {'MBP-I': 0, 'MBP-II': 1, 'MBP-III': 2}
s14_agreement_records = []
s14_confusion_records = []
for endpoint, excl_col in S14_ENDPOINT_DEFINITIONS.items():
    d = s14_mru[['Primary_MBP', excl_col]].dropna().copy()
    primary_num = d['Primary_MBP'].map(MBP_NUMERIC_S14)
    excl_num = d[excl_col].map(MBP_NUMERIC_S14)
    exact_agreement = float((d['Primary_MBP'] == d[excl_col]).mean())
    weighted_kappa = float(cohen_kappa_score(primary_num, excl_num, weights='quadratic'))
    mean_abs_shift = float(np.mean(np.abs(primary_num - excl_num)))
    s14_agreement_records.append({'Endpoint_excluded': endpoint, 'n': int(len(d)), 'Exact_agreement': exact_agreement, 'Quadratic_weighted_kappa': weighted_kappa, 'Mean_absolute_category_shift': mean_abs_shift})
    table = pd.crosstab(d['Primary_MBP'], d[excl_col]).reindex(index=['MBP-I', 'MBP-II', 'MBP-III'], columns=['MBP-I', 'MBP-II', 'MBP-III'], fill_value=0)
    for primary_grp in table.index:
        for excl_grp in table.columns:
            s14_confusion_records.append({'Endpoint_excluded': endpoint, 'Primary_MBP': primary_grp, 'Endpoint_excluded_MBP': excl_grp, 'n': int(table.loc[primary_grp, excl_grp])})
s14_agreement_df = pd.DataFrame(s14_agreement_records)
s14_confusion_df = pd.DataFrame(s14_confusion_records)
print('Figure S14 endpoint-excluded effect estimates:')

print('Figure S14 phenotype agreement:')



In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.7), facecolor='white')
endpoint_y_s14 = {'Response': 2, 'PFS': 1, 'OS': 0}
endpoint_color_s14 = {'Response': globals().get('ENDPOINT_COLORS', {}).get('Response', '#D95F59'), 'PFS': ENDPOINT_COLORS['PFS'], 'OS': ENDPOINT_COLORS['OS']}
ax.axvline(1.0, color='#888888', linestyle='--', linewidth=1.1, zorder=0)
for endpoint in ['Response', 'PFS', 'OS']:
    d = s14_effect_df[s14_effect_df['Endpoint'] == endpoint].copy()
    primary = d[d['Definition'] == 'Primary MBP'].iloc[0]
    excluded = d[d['Definition'].str.contains('excluded', case=False, regex=False)].iloc[0]
    y = endpoint_y_s14[endpoint]
    color = endpoint_color_s14[endpoint]
    ax.plot([float(primary['Estimate']), float(excluded['Estimate'])], [y, y], color='#B8B8B8', linewidth=1.8, zorder=1)
    ax.plot([float(primary['CI_lower']), float(primary['CI_upper'])], [y + 0.11, y + 0.11], color='#8A8A8A', linewidth=1.4, zorder=2)
    ax.scatter(float(primary['Estimate']), y + 0.11, s=47.5, facecolor='white', edgecolor='#707070', linewidth=1.2, zorder=3)
    ax.plot([float(excluded['CI_lower']), float(excluded['CI_upper'])], [y - 0.11, y - 0.11], color=color, linewidth=2, zorder=2)
    ax.scatter(float(excluded['Estimate']), y - 0.11, s=58.8, marker='D', facecolor=color, edgecolor='white', linewidth=0.8, zorder=4)
    p_excl = float(excluded['P'])
    p_txt = 'P<0.001' if p_excl < 0.001 else f'P={p_excl:.3f}'
    ax.text(float(excluded['CI_upper']) * 1.1, y - 0.11, f"{excluded['Estimate']:.2f} ({excluded['CI_lower']:.2f}–{excluded['CI_upper']:.2f}); {p_txt}", ha='left', va='center', fontsize=11.7, color=COLOR_TEXT)
all_ci_low_s14 = s14_effect_df['CI_lower'].replace([np.inf, -np.inf], np.nan).dropna()
all_ci_high_s14 = s14_effect_df['CI_upper'].replace([np.inf, -np.inf], np.nan).dropna()
xmin_s14a = max(0.18, float(all_ci_low_s14.min()) / 1.8)
xmax_s14a = max(7.0, float(all_ci_high_s14.max()) * 2.0)
ax.set_xscale('log')
ax.set_xlim(xmin_s14a, xmax_s14a)
ax.set_yticks([2, 1, 0])
ax.set_yticklabels(['mRECIST non-responder', 'PFS', 'OS'])
ax.set_xlabel('Adjusted effect per one-level phenotype worsening (OR / HR, log scale)', fontsize=15.2)
ax.tick_params(axis='y', length=0, pad=8)
ax.grid(False)
for spine in ['top', 'right', 'left']:
    ax.spines[spine].set_visible(False)
ax.legend(handles=[Line2D([0], [0], marker='o', linestyle='none', markerfacecolor='white', markeredgecolor='#707070', label='Primary MBP'), Line2D([0], [0], marker='D', linestyle='none', markerfacecolor='#555555', markeredgecolor='#555555', label='Target-endpoint-excluded MBP')], loc='upper left', frameon=False, fontsize=12.5)
fig.subplots_adjust(left=0.24, right=0.81, top=0.84, bottom=0.12)
FIGS14A_STEM = SUPP_OUT_DIR / get_value('science.paths.directory_090', 'directory_090')
fig.savefig(str(FIGS14A_STEM) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(FIGS14A_STEM) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(FIGS14A_STEM) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
from matplotlib.colors import Normalize
fig, axes = plt.subplots(1, 3, figsize=(9.0, 3.4), facecolor='white')
endpoint_order_s14b = ['Response', 'PFS', 'OS']
for ax, endpoint in zip(axes, endpoint_order_s14b):
    d = s14_confusion_df[s14_confusion_df['Endpoint_excluded'] == endpoint].copy()
    matrix = np.zeros((3, 3), dtype=float)
    for i, primary_grp in enumerate(['MBP-I', 'MBP-II', 'MBP-III']):
        for j, excl_grp in enumerate(['MBP-I', 'MBP-II', 'MBP-III']):
            row = d[(d['Primary_MBP'] == primary_grp) & (d['Endpoint_excluded_MBP'] == excl_grp)]
            if len(row) == 1:
                matrix[i, j] = float(row.iloc[0]['n'])
    row_sums = matrix.sum(axis=1, keepdims=True)
    pct = np.divide(matrix, row_sums, out=np.zeros_like(matrix), where=row_sums > 0)
    im = ax.imshow(pct, cmap='Blues', vmin=0, vmax=1, aspect='equal', interpolation='nearest')
    for i in range(3):
        for j in range(3):
            text_color = 'white' if pct[i, j] > 0.55 else COLOR_TEXT
            ax.text(j, i - 0.06, f'n={int(matrix[i, j])}', ha='center', va='center', fontsize=12.9, fontweight='bold', color=text_color)
            ax.text(j, i + 0.16, f'{100 * pct[i, j]:.0f}%', ha='center', va='center', fontsize=11.2, color=text_color)
    agreement_row = s14_agreement_df[s14_agreement_df['Endpoint_excluded'] == endpoint].iloc[0]
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['I', 'II', 'III'])
    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(['I', 'II', 'III'])
    ax.set_xlabel('Endpoint-excluded MBP', fontsize=13)
    if ax is axes[0]:
        ax.set_ylabel('Primary MBP', fontsize=13)
    ax.tick_params(length=0, labelsize=9.4)
    ax.set_xticks(np.arange(-0.5, 3, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 3, 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=2.1)
    for spine in ax.spines.values():
        spine.set_visible(False)
fig.subplots_adjust(left=0.08, right=0.98, top=0.78, bottom=0.11, wspace=0.28)
FIGS14B_STEM = SUPP_OUT_DIR / get_value('science.paths.directory_091', 'directory_091')
fig.savefig(str(FIGS14B_STEM) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(FIGS14B_STEM) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(FIGS14B_STEM) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
s14_mru.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_113', 'result_113.csv'), index=False, encoding='utf-8-sig')
s14_effect_df.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_114', 'result_114.csv'), index=False, encoding='utf-8-sig')
s14_agreement_df.to_csv(SOURCE_OUT_DIR / get_value('science.tables.metrics_054', 'metrics_054.csv'), index=False, encoding='utf-8-sig')
s14_confusion_df.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_115', 'result_115.csv'), index=False, encoding='utf-8-sig')



In [ ]:
from sklearn.metrics import cohen_kappa_score
from scipy.stats import spearmanr
import statsmodels.api as sm
from lifelines import CoxPHFitter
s15_source_cols = ['sample', 'Response_benefit', 'PFS_benefit', 'OS_benefit', 'MRU_benefit_score', 'MBP_raw']
missing_s15_source = [c for c in s15_source_cols if c not in mru_df.columns]
if missing_s15_source:
    raise KeyError(f'Run the primary MRU/MBP reconstruction cells first. Missing from mru_df: {missing_s15_source}')
s15_base = mru_df[s15_source_cols].drop_duplicates(subset=['sample']).copy()
s15_analysis_df = clinical_model_df.drop(columns=[c for c in ['Response_benefit', 'PFS_benefit', 'OS_benefit', 'MRU_benefit_score', 'MBP_raw'] if c in clinical_model_df.columns], errors='ignore').merge(s15_base, on='sample', how='left', validate='one_to_one')
if 'Nonresponse' not in s15_analysis_df.columns:
    s15_analysis_df['Nonresponse'] = 1 - pd.to_numeric(s15_analysis_df['mRECIST_R'], errors='coerce')
S15_ADJUSTERS = [c for c in ['ALBI_z', 'TBS_z', 'AFP_z', 'NLR_z', 'PVTT_model'] if c in s15_analysis_df.columns]
MBP_NUMERIC_S15 = {'MBP-I': 0.0, 'MBP-II': 1.0, 'MBP-III': 2.0}

def assign_tertile_from_score_s15(score):
    s = pd.to_numeric(score, errors='coerce')
    q1 = float(s.quantile(1 / 3))
    q2 = float(s.quantile(2 / 3))
    grp = pd.Series(np.nan, index=s.index, dtype=object)
    grp.loc[s >= q2] = 'MBP-I'
    grp.loc[(s >= q1) & (s < q2)] = 'MBP-II'
    grp.loc[s < q1] = 'MBP-III'
    return (grp, q1, q2)

def build_ordinal_categories_s15(score, n_groups):
    s = pd.to_numeric(score, errors='coerce')
    ranks = s.rank(method='first')
    low_to_high = pd.qcut(ranks, q=n_groups, labels=False, duplicates='drop')
    n_actual = int(pd.Series(low_to_high).dropna().nunique())
    if n_actual < 2:
        return (pd.Series(np.nan, index=s.index), n_actual)
    worse = (n_actual - 1 - low_to_high).astype(float)
    return (worse, n_actual)
S15_WEIGHT_SCHEMES = {'Equal weight (primary)': (1 / 3, 1 / 3, 1 / 3), 'Response emphasis': (0.5, 0.25, 0.25), 'PFS emphasis': (0.25, 0.5, 0.25), 'OS emphasis': (0.25, 0.25, 0.5)}
primary_group_s15 = s15_analysis_df['MBP_raw'].astype(object)
primary_numeric_s15 = primary_group_s15.map(MBP_NUMERIC_S15)
s15_weight_agreement_records = []
s15_weight_patient_cols = ['sample', 'MBP_raw', 'MRU_benefit_score']
for scheme_name, (wr, wp, wo) in S15_WEIGHT_SCHEMES.items():
    safe_name = scheme_name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    score_col = f'_s15_score_{safe_name}'
    group_col = f'_s15_group_{safe_name}'
    s15_analysis_df[score_col] = wr * s15_analysis_df['Response_benefit'] + wp * s15_analysis_df['PFS_benefit'] + wo * s15_analysis_df['OS_benefit']
    if scheme_name == 'Equal weight (primary)':
        s15_analysis_df[group_col] = primary_group_s15
        q1 = float(s15_analysis_df['MRU_benefit_score'].quantile(1 / 3))
        q2 = float(s15_analysis_df['MRU_benefit_score'].quantile(2 / 3))
    else:
        s15_analysis_df[group_col], q1, q2 = assign_tertile_from_score_s15(s15_analysis_df[score_col])
    s15_weight_patient_cols.extend([score_col, group_col])
    valid = primary_group_s15.notna() & s15_analysis_df[group_col].notna()
    primary_valid = primary_group_s15.loc[valid]
    perturbed_valid = s15_analysis_df.loc[valid, group_col]
    primary_num = primary_valid.map(MBP_NUMERIC_S15)
    perturbed_num = perturbed_valid.map(MBP_NUMERIC_S15)
    exact_agreement = float((primary_valid == perturbed_valid).mean())
    weighted_kappa = float(cohen_kappa_score(primary_num, perturbed_num, weights='quadratic'))
    n_changed = int((primary_valid != perturbed_valid).sum())
    rho, rho_p = spearmanr(pd.to_numeric(s15_analysis_df.loc[valid, 'MRU_benefit_score'], errors='coerce'), pd.to_numeric(s15_analysis_df.loc[valid, score_col], errors='coerce'))
    s15_weight_agreement_records.append({'Weight_scheme': scheme_name, 'n': int(valid.sum()), 'Exact_agreement': exact_agreement, 'Quadratic_weighted_kappa': weighted_kappa, 'n_changed': n_changed, 'Spearman_rho_vs_primary_score': float(rho), 'Spearman_P': float(rho_p), 'Q1_cutoff': q1, 'Q2_cutoff': q2, 'Response_weight': wr, 'PFS_weight': wp, 'OS_weight': wo})
s15_weight_agreement_df = pd.DataFrame(s15_weight_agreement_records)

def fit_adjusted_effect_s15(data, endpoint, predictor, definition, n_groups):
    if endpoint == 'Response':
        cols = ['Nonresponse', predictor] + S15_ADJUSTERS
        d = data[cols].dropna().copy()
        y = d['Nonresponse'].astype(int)
        X = sm.add_constant(d[[predictor] + S15_ADJUSTERS], has_constant='add')
        model = sm.GLM(y, X, family=sm.families.Binomial()).fit()
        beta = float(model.params[predictor])
        se = float(model.bse[predictor])
        return {'Endpoint': endpoint, 'Definition': definition, 'n_groups': int(n_groups), 'Measure': 'OR', 'Estimate': float(np.exp(beta)), 'CI_lower': float(np.exp(beta - 1.96 * se)), 'CI_upper': float(np.exp(beta + 1.96 * se)), 'P': float(model.pvalues[predictor]), 'n': int(len(d)), 'events': int(y.sum()), 'Interpretation': 'per one-category worsening'}
    time_col = f'{endpoint}_time'
    status_col = f'{endpoint}_status'
    cols = [time_col, status_col, predictor] + S15_ADJUSTERS
    d = data[cols].dropna().copy()
    d[status_col] = pd.to_numeric(d[status_col], errors='coerce').astype(int)
    cph = CoxPHFitter()
    try:
        cph.fit(d, duration_col=time_col, event_col=status_col, show_progress=False)
        penalizer_used = 0.0
    except Exception:
        cph = CoxPHFitter(penalizer=0.01)
        cph.fit(d, duration_col=time_col, event_col=status_col, show_progress=False)
        penalizer_used = 0.01
    s = cph.summary.loc[predictor]
    return {'Endpoint': endpoint, 'Definition': definition, 'n_groups': int(n_groups), 'Measure': 'HR', 'Estimate': float(s['exp(coef)']), 'CI_lower': float(s['exp(coef) lower 95%']), 'CI_upper': float(s['exp(coef) upper 95%']), 'P': float(s['p']), 'n': int(len(d)), 'events': int(d[status_col].sum()), 'penalizer': penalizer_used, 'Interpretation': 'per one-category worsening'}
S15_CUTOFF_SCHEMES = {'Median split': 2, 'Tertiles (primary)': 3, 'Quartiles': 4}
s15_cutoff_effect_records = []
s15_cutoff_patient_cols = ['sample']
for cutoff_name, requested_groups in S15_CUTOFF_SCHEMES.items():
    safe_name = cutoff_name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    ord_col = f'_s15_cutoff_{safe_name}'
    s15_analysis_df[ord_col], n_actual = build_ordinal_categories_s15(s15_analysis_df['MRU_benefit_score'], n_groups=requested_groups)
    s15_cutoff_patient_cols.append(ord_col)
    for endpoint in ['Response', 'PFS', 'OS']:
        s15_cutoff_effect_records.append(fit_adjusted_effect_s15(s15_analysis_df, endpoint=endpoint, predictor=ord_col, definition=cutoff_name, n_groups=n_actual))
s15_cutoff_effect_df = pd.DataFrame(s15_cutoff_effect_records)
print('S15A weight-perturbation stability:')

print('S15B cutoff sensitivity, per one-category worsening:')



In [ ]:
from matplotlib.lines import Line2D
scheme_order_s15a = ['Equal weight (primary)', 'Response emphasis', 'PFS emphasis', 'OS emphasis']
scheme_display_s15a = {'Equal weight (primary)': 'Equal weight (primary)', 'Response emphasis': 'Responder emphasis', 'PFS emphasis': 'PFS emphasis', 'OS emphasis': 'OS emphasis'}
scheme_colors_s15a = {'Equal weight (primary)': '#666666', 'Response emphasis': globals().get('ENDPOINT_COLORS', {}).get('Response', '#D95F59'), 'PFS emphasis': ENDPOINT_COLORS['PFS'], 'OS emphasis': ENDPOINT_COLORS['OS']}
plot_s15a = s15_weight_agreement_df.set_index('Weight_scheme').loc[scheme_order_s15a].reset_index()
ypos_s15a = {scheme: len(scheme_order_s15a) - 1 - i for i, scheme in enumerate(scheme_order_s15a)}
fig, ax = plt.subplots(figsize=(7.6, 4.2), facecolor='white')
ax.axvline(1.0, color='#B5B5B5', linestyle=':', linewidth=1.2, zorder=0)
for _, row in plot_s15a.iterrows():
    scheme = row['Weight_scheme']
    y = ypos_s15a[scheme]
    exact = float(row['Exact_agreement'])
    kappa = float(row['Quadratic_weighted_kappa'])
    changed = int(row['n_changed'])
    rho = float(row['Spearman_rho_vs_primary_score'])
    color = scheme_colors_s15a[scheme]
    ax.plot([min(exact, kappa), max(exact, kappa)], [y, y], color='#C7C7C7', linewidth=2.1, zorder=1)
    ax.scatter(exact, y, s=72.5, marker='o', facecolor=color, edgecolor='white', linewidth=0.9, zorder=3)
    ax.scatter(kappa, y, s=66.2, marker='D', facecolor='white', edgecolor=color, linewidth=1.5, zorder=4)
    ax.text(1.012, y, f'{100 * exact:.1f}% agree | {changed} changed | ρ={rho:.3f}', ha='left', va='center', fontsize=12.1, color=COLOR_TEXT, clip_on=False)
min_metric_s15a = float(min(plot_s15a['Exact_agreement'].min(), plot_s15a['Quadratic_weighted_kappa'].min()))
xmin_s15a = max(0.5, np.floor((min_metric_s15a - 0.06) * 20) / 20)
ax.set_xlim(xmin_s15a, 1.115)
ax.set_ylim(-0.55, len(scheme_order_s15a) - 0.45)
ax.set_yticks([ypos_s15a[x] for x in scheme_order_s15a])
ax.set_yticklabels([scheme_display_s15a[x] for x in scheme_order_s15a])
for tick, scheme in zip(ax.get_yticklabels(), scheme_order_s15a):
    tick.set_color(scheme_colors_s15a[scheme])
ax.set_xlabel('Agreement with primary equal-weight MBP', fontsize=15.5)
ax.tick_params(axis='y', length=0, pad=8)
ax.grid(False)
for y in ypos_s15a.values():
    ax.axhline(y, color=COLOR_GRID, linewidth=0.6, zorder=0)
for spine in ['top', 'right', 'left']:
    ax.spines[spine].set_visible(False)
ax.legend(handles=[Line2D([0], [0], marker='o', linestyle='none', markerfacecolor='#666666', markeredgecolor='white', markersize=7.9, label='Exact agreement'), Line2D([0], [0], marker='D', linestyle='none', markerfacecolor='white', markeredgecolor='#666666', markersize=7.3, label='Quadratic weighted κ')], loc='lower left', frameon=False, fontsize=12.5, ncol=2)
fig.subplots_adjust(left=0.23, right=0.79, top=0.83, bottom=0.11)
FIGS15A_STEM = SUPP_OUT_DIR / get_value('science.paths.directory_092', 'directory_092')
fig.savefig(str(FIGS15A_STEM) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(FIGS15A_STEM) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(FIGS15A_STEM) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
cutoff_order_s15b = ['Median split', 'Tertiles (primary)', 'Quartiles']
endpoint_offset_s15b = {'Response': 0.18, 'PFS': 0.0, 'OS': -0.18}
endpoint_colors_s15b = {'Response': globals().get('ENDPOINT_COLORS', {}).get('Response', '#D95F59'), 'PFS': ENDPOINT_COLORS['PFS'], 'OS': ENDPOINT_COLORS['OS']}
y_base_s15b = {definition: len(cutoff_order_s15b) - 1 - i for i, definition in enumerate(cutoff_order_s15b)}
fig, ax = plt.subplots(figsize=(7.85, 4.9), facecolor='white')
ax.axvline(1.0, color='#888888', linestyle='--', linewidth=1.1, zorder=0)
for _, row in s15_cutoff_effect_df.iterrows():
    definition = row['Definition']
    endpoint = row['Endpoint']
    y = y_base_s15b[definition] + endpoint_offset_s15b[endpoint]
    est = float(row['Estimate'])
    lo = float(row['CI_lower'])
    hi = float(row['CI_upper'])
    p = float(row['P'])
    color = endpoint_colors_s15b[endpoint]
    ax.plot([lo, hi], [y, y], color=color, linewidth=1.9, zorder=2)
    ax.scatter(est, y, s=50, facecolor=color, edgecolor='white', linewidth=0.8, zorder=3)
    p_txt = 'P<0.001' if p < 0.001 else f'P={p:.3f}'
    ax.text(hi * 1.08, y, f'{est:.2f} ({lo:.2f}–{hi:.2f}); {p_txt}', ha='left', va='center', fontsize=11.4, color=COLOR_TEXT)
for definition in cutoff_order_s15b:
    ax.axhline(y_base_s15b[definition], color=COLOR_GRID, linewidth=0.6, zorder=0)
lo_all_s15b = s15_cutoff_effect_df['CI_lower'].replace([np.inf, -np.inf], np.nan).dropna()
hi_all_s15b = s15_cutoff_effect_df['CI_upper'].replace([np.inf, -np.inf], np.nan).dropna()
ax.set_xscale('log')
ax.set_xlim(max(0.25, float(lo_all_s15b.min()) / 1.7), max(8.0, float(hi_all_s15b.max()) * 2.1))
ax.set_yticks([y_base_s15b[x] for x in cutoff_order_s15b])
ax.set_yticklabels(cutoff_order_s15b)
ax.set_xlabel('Adjusted effect per one-category MBP worsening (OR / HR, log scale)', fontsize=15.1)
ax.tick_params(axis='y', length=0, pad=8)
ax.grid(False)
for spine in ['top', 'right', 'left']:
    ax.spines[spine].set_visible(False)
ax.legend(handles=[Line2D([0], [0], marker='o', color=endpoint_colors_s15b['Response'], markerfacecolor=endpoint_colors_s15b['Response'], linewidth=1.8, label='Non-Responder OR'), Line2D([0], [0], marker='o', color=endpoint_colors_s15b['PFS'], markerfacecolor=endpoint_colors_s15b['PFS'], linewidth=1.8, label='PFS HR'), Line2D([0], [0], marker='o', color=endpoint_colors_s15b['OS'], markerfacecolor=endpoint_colors_s15b['OS'], linewidth=1.8, label='OS HR')], loc='upper left', frameon=False, fontsize=12.3, ncol=3)
fig.subplots_adjust(left=0.22, right=0.8, top=0.84, bottom=0.12)
FIGS15B_STEM = SUPP_OUT_DIR / get_value('science.paths.directory_093', 'directory_093')
fig.savefig(str(FIGS15B_STEM) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
fig.savefig(str(FIGS15B_STEM) + '.pdf', bbox_inches='tight', facecolor='white')
fig.savefig(str(FIGS15B_STEM) + '.svg', bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
s15_weight_agreement_df.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_116', 'result_116.csv'), index=False, encoding='utf-8-sig')
s15_cutoff_effect_df.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_117', 'result_117.csv'), index=False, encoding='utf-8-sig')
s15_analysis_df[list(dict.fromkeys(s15_weight_patient_cols + s15_cutoff_patient_cols + ['MRU_benefit_score']))].to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_118', 'result_118.csv'), index=False, encoding='utf-8-sig')



In [ ]:
import pandas as pd
from pathlib import Path
TABLE_DIR = Path(get_value('science.paths.directory_107', 'directory_107'))
TABLE_DIR.mkdir(exist_ok=True)
OUTPUT = TABLE_DIR / get_value('science.tables.result_188', 'result_188.xlsx')

def find_df(keyword_list):
    candidates = {}
    for name, obj in globals().items():
        if isinstance(obj, pd.DataFrame):
            low = name.lower()
            if any((k.lower() in low for k in keyword_list)):
                candidates[name] = obj
    return candidates
exported = {}
assoc = find_df(['cox', 'hazard', 'hr', 'association', 'multivariable', 'univariable', 'regression', 'model_summary'])
if assoc:
    exported['S5_MBP_association'] = pd.concat([df.assign(source=name) for name, df in assoc.items()], ignore_index=True)
phenotype = find_df(['phenotype', 'baseline', 'clinical', 'mbp_group', 'group_summary'])
if phenotype:
    exported['S6_MBP_phenotype'] = pd.concat([df.assign(source=name) for name, df in phenotype.items()], ignore_index=True)
robust = find_df(['robust', 'sensitivity', 'cutoff', 'weight'])
if robust:
    exported['S6_MBP_robustness'] = pd.concat([df.assign(source=name) for name, df in robust.items()], ignore_index=True)
with pd.ExcelWriter(OUTPUT, engine='openpyxl', mode='a' if OUTPUT.exists() else 'w') as writer:
    for sheet, df in exported.items():
        sheet = sheet[:31]
        df.to_excel(writer, sheet_name=sheet, index=False)


for k, v in exported.items():
    print(k, v.shape)
